# Zero-shot Generation - End-to-End Evaluation Pipeline

This notebook implements the **evaluation side** of the Zero-shot Generation.

### Notebook Structure
Given a directory of model generations, it runs a full pipeline:

1. **LilyPond → MIDI rendering**  
   - Wraps raw LilyPond snippets into a valid `\score` block if needed.  
   - Calls the system LilyPond binary, captures logs, and writes a cleaned `.ly` with `\midi` and `\layout` blocks.

2. **MIDI-level musical analysis (music21)**  
   - Parses the rendered MIDI and computes structural and musical descriptors: note count, range, interval entropy, step vs. leap, bar structure, key adherence, tonal stability, contour stats, etc.

3. **LilyPond text-level evaluation**  
   - Inspects the LilyPond source directly for adherence to our prompt constraints: relative/absolute notation, lowercase, forbidden constructs, bar structure, final barline, etc.  

4. **Batch aggregation & reporting**  
   - For each batch, writes JSONL logs per stage, a combined `final/index.jsonl`, and an `AVG_row.json` summary.  
   - Builds a human-readable Pandas table per batch, with an extra **AVG** row summarizing compile rates, in-key percentage, style label distribution (scalar vs balanced), and other metrics.

### Requirements
- **Python 3.10+**
- **LilyPond 2.24.4** installed and reachable from:
  - PATH, or  
  - `LILYPOND_BIN` environment variable, or  
  - one of the default Windows locations.
- **Python libraries:**
  - `music21`
  - `pandas`
  - `IPython`

### Output
The goal is to treat this notebook as the **single entry point** for evaluating prompt configurations: once generations are saved under `zero_shot_outputs/raw/`, we can run this file to obtain both per-sample diagnostics and batch-level statistics ready for analysis.


## Block 1 - LilyPond → MIDI renderer (`lily_to_midi`)

This block is responsible for taking a **raw LilyPond snippet** (as generated by the model) and turning it into a **MIDI file**.

Main responsibilities:

- **LilyPond binary discovery**
  - `find_lilypond()` searches for a usable LilyPond executable via PATH, `LILYPOND_BIN`, or a Windows default path.
  - `_is_working_lilypond()` sanity-checks the executable by calling `--version`.

- **Score wrapping / sanitization**
  - `_extract_version_line()` pulls out an existing `\version` line so it can be preserved.
  - `_pull_block_by_name()` extracts and reuses `\header` and `\paper` blocks if they exist.
  - `_hoist_top_level_assignments()` moves top-level variable assignments (like `mySeq = { ... }`) to the top of the file so they are visible inside the score.
  - `_has_top_level_staff()` decides whether to wrap the music in a `\new Staff` or reuse existing staff contexts.
  - `_build_score_from_body()` assembles a complete `\score { ... }` block, and ensures that `\layout {}` and `\midi {}` sub-blocks are present.

- **Rendering to MIDI**
  - `lily_to_midi()`:
    - Derives an output MIDI filename from the input `.ly`.
    - Caches and reuses an existing MIDI file if `force=False` and the target already exists.
    - Writes a cleaned/augmented `.ly` (`ly_with_midi_*.ly`).
    - Calls LilyPond in a temporary directory, then moves the produced MIDI into `midi_dir`.
    - Writes a detailed `.log` file on failure, including command line and LilyPond stdout/stderr.

- **Return structure**
  - Returns a structured dict with:
    - `ok` flag and `reason` (`rendered`, `exists`, `encoding_error`, `lilypond_failed`, `no_midi_output`).
    - Timing information (`seconds`).
    - Paths to the MIDI and `ly_with_midi` files.
    - SHA-256 hash of the original `.ly` source.
    - Truncated error message for logging.

This block is purely about **getting a MIDI file out of whatever LilyPond the model produced**, while being as tolerant as possible to partial or messy input.


In [3]:
from __future__ import annotations
from pathlib import Path
from typing import Dict, Any, Optional, Tuple, List
import subprocess
import tempfile
import shutil
import hashlib
import time
import os
import re

LILYPOND_TIMEOUT_SECONDS = 5
MAX_ERROR_TEXT_LENGTH = 800

DEFAULT_WINDOWS_LILYPOND = Path(
    r"C:\lilypond-2.24.4-mingw-x86_64\lilypond-2.24.4\bin\lilypond.exe"
)

_VERSION_RE = re.compile(r'^\s*\\version\b.*$', re.MULTILINE)
_ASSIGN_START = re.compile(r'(?m)^[A-Za-z_][A-Za-z0-9_-]*\s*=\s*')


def _is_working_lilypond(executable: Path | str) -> bool:
    """Check whether the given path is a valid, runnable LilyPond executable."""
    try:
        subprocess.run(
            [str(executable), "--version"],
            capture_output=True,
            timeout=LILYPOND_TIMEOUT_SECONDS,
            check=True,
        )
        return True
    except Exception:
        return False


def find_lilypond() -> Path:
    """Locate a usable LilyPond executable through PATH, env vars, or defaults."""
    lily_name = "lilypond.exe" if os.name == "nt" else "lilypond"
    path_candidate = shutil.which(lily_name)

    if path_candidate and _is_working_lilypond(path_candidate):
        return Path(path_candidate)

    env_candidate = os.environ.get("LILYPOND_BIN")
    if env_candidate and Path(env_candidate).exists():
        if _is_working_lilypond(env_candidate):
            return Path(env_candidate)

    if DEFAULT_WINDOWS_LILYPOND.exists():
        if _is_working_lilypond(DEFAULT_WINDOWS_LILYPOND):
            return DEFAULT_WINDOWS_LILYPOND

    raise FileNotFoundError("LilyPond executable not found.")


def _extract_version_line(text: str) -> Tuple[Optional[str], str]:
    """Extract the \\version line if present and return (version_line, text_without_it)."""
    match = _VERSION_RE.search(text)
    if not match:
        return None, text

    version_line = match.group(0).strip()
    start, end = match.span()
    remaining = text[:start] + text[end:]
    remaining = re.sub(r'^\s*\n', '', remaining, count=1, flags=re.MULTILINE)
    return version_line, remaining


def _sha256_text(text: str) -> str:
    """Return SHA-256 digest of a UTF-8 string."""
    digest = hashlib.sha256()
    digest.update(text.encode("utf-8"))
    return digest.hexdigest()


def _truncate(text: Optional[str], max_length: int = MAX_ERROR_TEXT_LENGTH) -> Optional[str]:
    """Trim long text for logging/debug output."""
    if not text:
        return text

    stripped = text.strip()
    if len(stripped) <= max_length:
        return stripped
    return stripped[:max_length] + "…"


def _derive_midi_name_from_out_ly(out_ly: Path) -> str:
    """Create a MIDI file stem based on the input .ly filename."""
    stem = out_ly.stem
    if stem.startswith("out_"):
        return "midi_" + stem[len("out_"):]
    return stem + "_midi"


def _pull_block_by_name(name: str, text: str) -> Tuple[str, List[str]]:
    """Extract blocks like \\header { ... } or \\score { ... } from text."""
    pattern = re.compile(rf'\\{name}\s*\{{', re.IGNORECASE)
    remaining = text
    blocks: List[str] = []
    search_start = 0

    while True:
        match = pattern.search(remaining, search_start)
        if not match:
            break

        depth = 0
        brace_index = match.end() - 1
        end_index: Optional[int] = None

        while brace_index < len(remaining):
            char = remaining[brace_index]
            if char == '{':
                depth += 1
            elif char == '}':
                depth -= 1
                if depth == 0:
                    end_index = brace_index + 1
                    break
            brace_index += 1

        if end_index is None:
            break

        blocks.append(remaining[match.start():end_index])
        remaining = remaining[:match.start()] + remaining[end_index:]
        search_start = match.start()

    return remaining, blocks


def _ensure_token_block(block: str, name: str) -> str:
    """Ensure a sub-block (like \\midi or \\layout) exists inside a score block."""
    if re.search(rf'\\{name}\s*\{{', block, flags=re.IGNORECASE):
        return block

    insert_position = block.rfind('}')
    if insert_position == -1:
        return block + f"\n  \\{name} {{}}\n"

    return (
        block[:insert_position]
        + f"\n  \\{name} {{}}\n"
        + block[insert_position:]
    )


def _strip_comments(text: str) -> str:
    """Remove LilyPond % comments."""
    return re.sub(r'%.*$', '', text, flags=re.MULTILINE)


def _has_top_level_staff(text: str) -> bool:
    """Check whether the code already contains \\new Staff or Staff context blocks."""
    without_comments = _strip_comments(text)
    pattern = re.compile(r'\\(?:new\s+Staff|context\s+Staff)\s*\{', re.IGNORECASE)
    return bool(pattern.search(without_comments))


def _hoist_top_level_assignments(text: str) -> Tuple[str, str]:
    """Move variable assignments at top level to the beginning of the file."""
    index = 0
    length = len(text)

    body_parts: List[str] = []
    definitions: List[str] = []

    while index < length:
        match = _ASSIGN_START.search(text, index)
        if not match:
            body_parts.append(text[index:])
            break

        body_parts.append(text[index:match.start()])
        scan_index = match.end()

        while scan_index < length and text[scan_index].isspace():
            scan_index += 1

        brace_start = text.find('{', scan_index)
        if brace_start == -1:
            body_parts.append(text[match.start():])
            break

        depth = 0
        brace_index = brace_start
        end_index: Optional[int] = None

        while brace_index < length:
            char = text[brace_index]
            if char == '{':
                depth += 1
            elif char == '}':
                depth -= 1
                if depth == 0:
                    end_index = brace_index + 1
                    break
            brace_index += 1

        if end_index is None:
            body_parts.append(text[match.start():])
            break

        trailing_index = end_index
        while trailing_index < length and text[trailing_index] in (" ", "\t", "\r", "\n"):
            trailing_index += 1

        definitions.append(text[match.start():trailing_index])
        index = trailing_index

    body = "".join(body_parts)
    defs_text = "".join(definitions).strip()
    if defs_text:
        defs_text += "\n\n"

    return body, defs_text


def _build_score_from_body(full_text: str) -> str:
    """Assemble a complete LilyPond score, wrapping into a \\score block if missing."""
    version_line, remaining = _extract_version_line(full_text)
    remaining, header_blocks = _pull_block_by_name("header", remaining)
    remaining, paper_blocks = _pull_block_by_name("paper", remaining)
    header_and_paper = "".join(header_blocks + paper_blocks)

    # If a score block exists, clean it and reuse the first one
    remainder_without_scores, score_blocks = _pull_block_by_name("score", remaining)
    if score_blocks:
        first_score = score_blocks[0]
        first_score = _ensure_token_block(first_score, "layout")
        first_score = _ensure_token_block(first_score, "midi")
        header_prefix = (version_line + "\n\n") if version_line else ""
        return f"{header_prefix}{header_and_paper}\n{first_score}\n"

    body = remaining
    body, hoisted_defs = _hoist_top_level_assignments(body)
    body = body.strip()

    # Decide how to wrap the music
    if not body:
        staff_payload = "s1"
    else:
        if _has_top_level_staff(body):
            staff_payload = body
        else:
            staff_payload = f"\\new Staff {{\n{body}\n}}"

    score = f"""\\score {{
  {staff_payload}
  \\layout {{}}
  \\midi {{}}
}}"""

    header_prefix = (version_line + "\n\n") if version_line else ""
    return f"{header_prefix}{header_and_paper}{hoisted_defs}{score}\n"


def lily_to_midi(
    out_ly: str | Path,
    *,
    midi_dir: str | Path,
    force: bool = False,
    lilypond_version_tag: str = "2.24.4",
) -> Dict[str, Any]:
    """Render a LilyPond file to MIDI and return result metadata."""
    out_ly_path = Path(out_ly)
    midi_dir_path = Path(midi_dir)
    midi_dir_path.mkdir(parents=True, exist_ok=True)

    midi_stem = _derive_midi_name_from_out_ly(out_ly_path)
    target_midi_path = (midi_dir_path / f"{midi_stem}.mid").resolve()
    ly_with_midi_path = (
        midi_dir_path / f"ly_with_midi_{midi_stem.split('_')[-1]}.ly"
    ).resolve()
    log_path = (midi_dir_path / f"{midi_stem}.log").resolve()

    # Reuse existing MIDI if allowed
    if target_midi_path.exists() and not force:
        try:
            source_hash = _sha256_text(out_ly_path.read_text(encoding="utf-8"))
        except Exception:
            source_hash = None

        return {
            "ok": True,
            "seconds": 0.0,
            "reason": "exists",
            "paths": {"ly_with_midi": None, "midi": str(target_midi_path)},
            "sha256": {"source_ly": source_hash},
            "tooling": {"lilypond_version": lilypond_version_tag},
            "error": None,
        }

    try:
        original_text = out_ly_path.read_text(encoding="utf-8")
    except UnicodeDecodeError as exc:
        return {
            "ok": False,
            "seconds": None,
            "reason": "encoding_error",
            "paths": {"ly_with_midi": None, "midi": str(target_midi_path)},
            "sha256": {"source_ly": None},
            "tooling": {"lilypond_version": lilypond_version_tag},
            "error": _truncate(str(exc)),
        }

    render_text = _build_score_from_body(original_text)
    source_hash = _sha256_text(original_text)

    try:
        ly_with_midi_path.write_text(render_text, encoding="utf-8")
    except Exception:
        pass

    lilypond_executable = find_lilypond()

    with tempfile.TemporaryDirectory() as temp_dir:
        temp_path = Path(temp_dir)
        temp_ly_path = temp_path / "render.ly"
        temp_ly_path.write_text(render_text, encoding="utf-8")

        out_base = temp_path / "render"
        cmd = [
            str(lilypond_executable),
            "-dno-point-and-click",
            "-o",
            str(out_base),
            str(temp_ly_path),
        ]

        start_time = time.perf_counter()
        proc = None
        try:
            proc = subprocess.run(
                cmd,
                capture_output=True,
                text=True,
                timeout=LILYPOND_TIMEOUT_SECONDS,
            )
        except subprocess.TimeoutExpired as exc:
            elapsed = time.perf_counter() - start_time
            try:
                log = (
                    f"[lilypond_timeout] cmd: {' '.join(cmd)}\n"
                    f"timeout_seconds: {LILYPOND_TIMEOUT_SECONDS}\n\n"
                    f"STDOUT:\n{exc.stdout or ''}\n\n"
                    f"STDERR:\n{exc.stderr or ''}\n"
                )
                log_path.write_text(log, encoding="utf-8")
            except Exception:
                pass
            return {
                "ok": False,
                "seconds": round(elapsed, 4),
                "reason": "lilypond_timeout",
                "paths": {
                    "ly_with_midi": str(ly_with_midi_path),
                    "midi": str(target_midi_path),
                },
                "sha256": {"source_ly": source_hash},
                "tooling": {"lilypond_version": lilypond_version_tag},
                "error": _truncate(
                    f"LilyPond timed out after {LILYPOND_TIMEOUT_SECONDS} seconds."
                ),
            }
        elapsed = time.perf_counter() - start_time

        def _dump_log(tag: str, extra: str = "") -> None:
            """Write detailed execution logs for debugging."""
            try:
                log = (
                    f"[{tag}] cmd: {' '.join(cmd)}\n"
                    f"returncode: {proc.returncode}\n\n"
                    f"STDOUT:\n{proc.stdout or ''}\n\n"
                    f"STDERR:\n{proc.stderr or ''}\n"
                )
                if extra:
                    log += f"\n{extra}\n"
                log_path.write_text(log, encoding="utf-8")
            except Exception:
                pass

        if proc.returncode != 0:
            _dump_log("lilypond_failed")
            return {
                "ok": False,
                "seconds": elapsed,
                "reason": "lilypond_failed",
                "paths": {
                    "ly_with_midi": str(ly_with_midi_path),
                    "midi": str(target_midi_path),
                },
                "sha256": {"source_ly": source_hash},
                "tooling": {"lilypond_version": lilypond_version_tag},
                "error": _truncate(proc.stderr or proc.stdout or "LilyPond failed"),
            }

        # Search for generated MIDI
        candidates: List[Path] = []
        for candidate in [
            out_base.with_suffix(".midi"),
            out_base.with_suffix(".mid"),
        ]:
            if candidate.exists():
                candidates.append(candidate)

        if not candidates:
            candidates += sorted(temp_path.glob("render-*.midi"))
            candidates += sorted(temp_path.glob("render-*.mid"))

        if not candidates:
            candidates += sorted(temp_path.glob("*.midi"))
            candidates += sorted(temp_path.glob("*.mid"))

        if not candidates:
            _dump_log(
                "no_midi_output",
                extra=(
                    "No MIDI produced. Check generated score or musical content.\n"
                    f"Generated file: {ly_with_midi_path}"
                ),
            )
            return {
                "ok": False,
                "seconds": elapsed,
                "reason": "no_midi_output",
                "paths": {
                    "ly_with_midi": str(ly_with_midi_path),
                    "midi": str(target_midi_path),
                },
                "sha256": {"source_ly": source_hash},
                "tooling": {"lilypond_version": lilypond_version_tag},
                "error": "LilyPond did not produce MIDI.",
            }

        shutil.move(str(candidates[0]), str(target_midi_path))

    return {
        "ok": True,
        "seconds": round(elapsed, 4),
        "reason": "rendered",
        "paths": {
            "ly_with_midi": str(ly_with_midi_path),
            "midi": str(target_midi_path),
        },
        "sha256": {"source_ly": source_hash},
        "tooling": {"lilypond_version": lilypond_version_tag},
        "error": None,
    }


## Block 2 - MIDI-level musical analysis (`eval_midi`)

Once we have a MIDI file, this block uses **music21** to compute a rich set of musical descriptors.  
It treats the MIDI as the “ground truth performance” of the generated piece, independent of how the LilyPond text looks.

Main pieces:

- **Core analysis**
  - `_analyze_midi_with_music21()` loads the MIDI into music21, flattens the score, and separates:
    - Notes
    - Chords
    - Measures
  - It builds a time-ordered “melodic line” (notes + chord basses) and derives:
    - `intervals` between successive pitches
    - global pitch range (`min_midi`, `max_midi`)

- **Melodic / structural metrics**
  - `interval_entropy`: Shannon entropy of the interval distribution.
  - `step_vs_leap`: proportion of small steps (≤ 2 semitones) vs larger leaps.
  - `direction_changes`: number of times the melodic contour changes up ↔ down.
  - `repeat_rate`: proportion of repeated notes.
  - `avg_interval_size`: mean absolute interval size.
  - `note_density`: notes per bar (or per piece if bar info is missing).
  - `rhythmic_diversity`: count of distinct rhythmic values (quarterLength variants).

- **Bar / time-signature metrics**
  - `_compute_beats_per_bar_match_rate()` compares the sum of durations in each measure against the expected total from the declared time signature.
  - `_compute_expected_bars()` infers how many bars the piece should have and compares it to the observed bar count.
  - Derived booleans:
    - `beats_ok`: beats per bar mostly consistent with time signature.
    - `bars_ok_musical`: bar count consistent (within ±1) with expected length.

- **Key / tonal metrics**
  - `_compute_detected_key()` runs music21’s key analysis and normalizes the key string.
  - `_compute_in_key_share_and_cadence()`:
    - Measures the time spent in the declared key (time-weighted in-key percentage).
    - Checks whether the piece ends on the tonic.
    - Returns `in_key_pct_time_weighted`, `ends_on_tonic`, and a boolean `in_key_ok` based on a threshold.
  - `_compute_tonal_stability()`:
    - Splits the piece into segments.
    - Runs local key analysis per segment.
    - Reports a `tonal_stability_score` (how often segments agree), `key_changes`, and `drift_detected`.

- **Contour metrics**
  - `_compute_contour_analysis()` reduces the pitch line to up/down/same symbols and counts:
    - Dominant contour patterns,
    - `contour_diversity`,
    - `ascending_tendency` / `descending_tendency`,
    - `arch_shapes` and `downward_arches`.

- **Public API**
  - `eval_midi(midi_path, declared_key_pc_mode, declared_time, ...)`:
    - Wraps the call with timing and error handling.
    - Returns:
      - `ok` flag,
      - `seconds` (runtime),
      - `metrics` dict (all numbers above),
      - `error` string and `tooling` (music21 version).

This block gives us a **high-level musical profile** of each generation, which we later aggregate per batch (compile rates, scalar vs balanced style, stability, etc.).


In [4]:
from __future__ import annotations

from pathlib import Path
from typing import Optional, Tuple, Dict, Any, List
from collections import defaultdict
import math
from collections import Counter

# Global thresholds and defaults for analysis behaviour
MIN_NOTES_FOR_TONAL_ANALYSIS = 6
MIN_SEGMENT_NOTES = 3
DEFAULT_SEGMENT_COUNT = 4
DEFAULT_IN_KEY_THRESHOLD = 0.98
DEFAULT_BEATS_EPSILON = 1e-6
MAX_ERROR_TEXT_LENGTH = 300


def _truncate_err(message: str | None, max_len: int = MAX_ERROR_TEXT_LENGTH) -> str | None:
    """Shorten an error message to a bounded length, keeping None untouched."""
    if not message:
        return message
    message = message.strip()
    return (message[:max_len] + "…") if len(message) > max_len else message


def _normalize_key_string(key: str | None) -> str | None:
    """Normalize a music21 key string to lowercase 'tonic mode' (e.g. 'c major')."""
    if not isinstance(key, str):
        return key

    parts = key.strip().split()
    if len(parts) == 2:
        return f"{parts[0].lower()} {parts[1].lower()}"
    return key.lower()


def _scale_pcset(tonic_pc: int, mode: str) -> set[int]:
    """Return the pitch-class set for a diatonic scale given tonic and mode."""
    mode_intervals = {
        "ionian":      [0, 2, 4, 5, 7, 9, 11],
        "major":       [0, 2, 4, 5, 7, 9, 11],
        "aeolian":     [0, 2, 3, 5, 7, 8, 10],
        "minor":       [0, 2, 3, 5, 7, 8, 10],
        "dorian":      [0, 2, 3, 5, 7, 9, 10],
        "phrygian":    [0, 1, 3, 5, 7, 8, 10],
        "lydian":      [0, 2, 4, 6, 7, 9, 11],
        "mixolydian":  [0, 2, 4, 5, 7, 9, 10],
        "locrian":     [0, 1, 3, 5, 6, 8, 10],
    }
    intervals = mode_intervals.get(mode.lower(), mode_intervals["major"])
    return {(tonic_pc + interval) % 12 for interval in intervals}


def _compute_tonal_stability(stream, declared_key_pc_mode: Optional[Tuple[int, str]] = None) -> Dict[str, Any]:
    """
    Estimate how stable the perceived key is across the piece,
    optionally relative to a declared key.
    """
    try:
        flat_stream = stream.flatten()

        notes_and_chords = [
            n for n in flat_stream.notes
            if getattr(n, "isNote", False) or getattr(n, "isChord", False)
        ]

        if len(notes_and_chords) < MIN_NOTES_FOR_TONAL_ANALYSIS:
            return {"tonal_stability_score": None, "key_changes": None, "drift_detected": None}

        segment_size = len(notes_and_chords) // DEFAULT_SEGMENT_COUNT
        if segment_size < MIN_SEGMENT_NOTES:
            segment_size = max(MIN_SEGMENT_NOTES, len(notes_and_chords) // 2)

        from music21 import stream as m21_stream
        segments: List[Dict[str, Any]] = []

        # Build short segments and run local key analysis
        for i in range(0, len(notes_and_chords), segment_size):
            segment_notes = notes_and_chords[i:i + segment_size]
            if len(segment_notes) < MIN_SEGMENT_NOTES:
                continue

            try:
                temp_stream = m21_stream.Stream()
                for note_or_chord in segment_notes:
                    temp_stream.append(note_or_chord)

                key_analysis = temp_stream.analyze("key")
                segments.append(
                    {
                        "index": len(segments),
                        "note_count": len(segment_notes),
                        "key": str(key_analysis),
                        "tonic_pc": key_analysis.tonic.midi % 12,
                        "mode": key_analysis.mode,
                    }
                )
            except Exception:
                # If one segment fails analysis, just ignore it
                continue

        if len(segments) < 2:
            return {"tonal_stability_score": None, "key_changes": None, "drift_detected": None}

        # If no declared key, measure everything relative to the first segment
        if declared_key_pc_mode is None:
            key_changes = 0
            previous_tonic = segments[0]["tonic_pc"]

            for segment in segments[1:]:
                if segment["tonic_pc"] != previous_tonic:
                    key_changes += 1
                previous_tonic = segment["tonic_pc"]

            stable_segments = sum(
                1
                for segment in segments
                if segment["tonic_pc"] == segments[0]["tonic_pc"]
            )
            stability_score = stable_segments / len(segments)
            drift_detected = (stability_score < 0.7) or (key_changes > 1)

            return {
                "tonal_stability_score": float(stability_score),
                "key_changes": int(key_changes),
                "drift_detected": bool(drift_detected),
            }

        # With a declared key, check agreement and count changes vs that key
        tonic_pc, mode = declared_key_pc_mode
        same_key_count = 0
        key_changes = 0
        previous_tonic = segments[0]["tonic_pc"]

        for segment in segments:
            if (
                segment["tonic_pc"] == tonic_pc
                and segment["mode"].lower() == mode.lower()
            ):
                same_key_count += 1

            if segment["tonic_pc"] != previous_tonic:
                key_changes += 1
            previous_tonic = segment["tonic_pc"]

        stability_score = same_key_count / len(segments)
        drift_detected = (stability_score < 0.7) or (key_changes > 1)

        return {
            "tonal_stability_score": float(stability_score),
            "key_changes": int(key_changes),
            "drift_detected": bool(drift_detected),
        }

    except Exception:
        # On failure, return a fully populated but empty-like structure
        return {"tonal_stability_score": None, "key_changes": None, "drift_detected": None}


def _compute_contour_analysis(sequential_pitches: List[int]) -> Dict[str, Any]:
    """
    Extract simple contour statistics (up/down patterns, arches, tendencies)
    from a sequence of MIDI pitches.
    """
    if len(sequential_pitches) < 4:
        return {
            "contour_types": [],
            "contour_diversity": 0,
            "ascending_tendency": 0.0,
            "descending_tendency": 0.0,
            "arch_shapes": 0,
            "downward_arches": 0,
        }

    contour_symbols: List[str] = []
    for i in range(1, len(sequential_pitches)):
        diff = sequential_pitches[i] - sequential_pitches[i - 1]
        if diff > 0:
            contour_symbols.append("U")
        elif diff < 0:
            contour_symbols.append("D")
        else:
            contour_symbols.append("S")

    contour_patterns: List[str] = []
    pattern_counts: Dict[str, int] = defaultdict(int)

    for i in range(len(contour_symbols) - 1):
        pattern = "".join(contour_symbols[i:i + 2])
        contour_patterns.append(pattern)
        pattern_counts[pattern] += 1

    directional_moves = [s for s in contour_symbols if s != "S"]
    total_moves = len(directional_moves)
    ascending_tendency = (
        contour_symbols.count("U") / max(1, total_moves)
    )
    descending_tendency = (
        contour_symbols.count("D") / max(1, total_moves)
    )

    arch_shapes = 0
    downward_arches = 0
    for i in range(len(contour_patterns) - 1):
        if contour_patterns[i] == "UD":
            arch_shapes += 1
        elif contour_patterns[i] == "DU":
            downward_arches += 1

    contour_diversity = len(pattern_counts) / max(1, len(contour_patterns))

    common_contours = sorted(
        pattern_counts.items(),
        key=lambda item: item[1],
        reverse=True,
    )[:3]
    contour_types = [pattern for pattern, _count in common_contours]

    return {
        "contour_types": contour_types,
        "contour_diversity": float(contour_diversity),
        "ascending_tendency": float(ascending_tendency),
        "descending_tendency": float(descending_tendency),
        "arch_shapes": int(arch_shapes),
        "downward_arches": int(downward_arches),
    }


def _compute_interval_entropy(intervals: List[int]) -> float:
    """Shannon entropy of the interval distribution (in semitones)."""
    if not intervals:
        return 0.0

    counts = Counter(intervals)
    total = sum(counts.values())
    entropy = 0.0

    for count in counts.values():
        p = count / total
        entropy -= p * math.log2(p)

    return float(entropy)


def _build_pitch_sequences(sf, notes, chords) -> Tuple[List[int], List[Tuple[str, Any]]]:
    """
    Build:
      - a flat list of all pitches (notes + chord tones),
      - a time-ordered sequence of ('note'|'chord', element).
    """
    all_pitches: List[int] = []
    for n in notes:
        all_pitches.append(int(n.pitch.midi))

    for chord in chords:
        for pitch in chord.pitches:
            all_pitches.append(int(pitch.midi))

    all_elements: List[Tuple[str, Any]] = []
    for n in notes:
        all_elements.append(("note", n))
    for chord in chords:
        all_elements.append(("chord", chord))

    all_elements.sort(key=lambda pair: pair[1].offset)

    return all_pitches, all_elements


def _compute_intervals_from_elements(all_elements: List[Tuple[str, Any]]) -> Tuple[List[int], List[int]]:
    """
    Reduce a mixed note/chord stream to a single pitch line and its intervals.
    Chords are represented by their lowest pitch.
    """
    sequential_pitches: List[int] = []

    for elem_type, elem in all_elements:
        if elem_type == "note":
            sequential_pitches.append(int(elem.pitch.midi))
        else:
            chord_pitches = [int(p.midi) for p in elem.pitches]
            sequential_pitches.append(min(chord_pitches))

    intervals = [
        sequential_pitches[i + 1] - sequential_pitches[i]
        for i in range(len(sequential_pitches) - 1)
    ]
    return sequential_pitches, intervals


def _compute_beats_per_bar_match_rate(
    s,
    measures,
    declared_time: Optional[Tuple[int, int]],
    beats_epsilon: float,
) -> Optional[float]:
    """Compute how many measures match the expected total duration from the time signature."""
    if not declared_time or not measures:
        return None

    num, den = declared_time
    target_quarters = num * (4.0 / den)

    ok_count = 0
    for measure in measures:
        elements = list(measure.flatten().notesAndRests)
        qsum = sum(
            float(getattr(e.duration, "quarterLength", 0.0))
            for e in elements
        )
        if abs(qsum - target_quarters) < max(
            beats_epsilon, beats_epsilon * target_quarters
        ):
            ok_count += 1

    if len(measures) == 0:
        return None

    return ok_count / len(measures)


def _compute_detected_key(s) -> Optional[str]:
    """Run music21 key analysis and normalize the resulting key name."""
    try:
        detected_key_raw = str(s.analyze("key"))
    except Exception:
        detected_key_raw = None
    return _normalize_key_string(detected_key_raw)


def _compute_in_key_share_and_cadence(
    notes,
    chords,
    sequential_pitches: List[int],
    declared_key_pc_mode: Optional[Tuple[int, str]],
    in_key_threshold: float,
) -> Tuple[Optional[float], Optional[bool], Optional[bool]]:
    """
    Measure time spent in the declared key and whether the piece ends on the tonic.

    Returns:
        (in_key_time_share, ends_on_tonic, in_key_ok)
    """
    if declared_key_pc_mode is None:
        return None, None, None

    tonic_pc, mode = declared_key_pc_mode
    pcset = _scale_pcset(tonic_pc, mode)

    total_q = 0.0
    in_q = 0.0

    for n in notes:
        q = float(getattr(n.duration, "quarterLength", 0.0))
        total_q += q
        if int(n.pitch.midi) % 12 in pcset:
            in_q += q

    for chord in chords:
        q = float(getattr(chord.duration, "quarterLength", 0.0))
        total_q += q
        chord_pcs = {int(p.midi) % 12 for p in chord.pitches}
        if chord_pcs.issubset(pcset):
            in_q += q

    if total_q <= 0:
        in_key_time_share = None
    else:
        in_key_time_share = in_q / total_q

    in_key_ok = (
        (in_key_time_share is not None)
        and (in_key_time_share >= in_key_threshold)
    )

    ends_on_tonic = None
    if sequential_pitches:
        ends_on_tonic = (
            sequential_pitches[-1] % 12 == (tonic_pc % 12)
        )

    return in_key_time_share, ends_on_tonic, in_key_ok


def _compute_expected_bars(
    sf,
    declared_time: Optional[Tuple[int, int]],
    bar_count: int,
    all_pitches: List[int],
) -> int:
    """
    Estimate how many bars the piece should have, using duration and time signature,
    with a fallback heuristic if needed.
    """
    notesAnd_rests = [
        ev for ev in sf.notesAndRests
        if getattr(ev, "duration", None)
    ]
    total_quarters = sum(
        float(getattr(ev.duration, "quarterLength", 0.0))
        for ev in notesAnd_rests
    )

    if declared_time and total_quarters > 0:
        num, den = declared_time
        target_quarters = num * (4.0 / den)
        if target_quarters > 0:
            return max(1, int(round(total_quarters / target_quarters)))

    if bar_count > 0:
        return bar_count

    return max(1, len(all_pitches) // 4)


def _compute_rhythmic_diversity(notes, chords) -> int:
    """Count the number of distinct rhythmic values used (in quarter lengths)."""
    rhythmic_values = set()
    for n in notes:
        rhythmic_values.add(n.duration.quarterLength)
    for chord in chords:
        rhythmic_values.add(chord.duration.quarterLength)
    return len(rhythmic_values)


def _compute_range_ok(
    all_pitches: List[int],
    range_bounds: Optional[Tuple[int, int]],
) -> Optional[bool]:
    """Return True if all pitches lie inside a given MIDI range, else None if unbounded."""
    if range_bounds is None or not all_pitches:
        return None

    low, high = range_bounds
    return (min(all_pitches) >= low) and (max(all_pitches) <= high)


def _analyze_midi_with_music21(
    midi_path: Path,
    *,
    declared_key_pc_mode: Optional[Tuple[int, str]] = None,
    declared_time: Optional[Tuple[int, int]] = None,
    range_bounds: Optional[Tuple[int, int]] = None,
    in_key_threshold: float = DEFAULT_IN_KEY_THRESHOLD,
    beats_epsilon: float = DEFAULT_BEATS_EPSILON,
) -> Dict[str, Any]:
    """
    Core analysis routine: parse a MIDI file with music21 and compute
    melodic, harmonic, rhythmic and tonal descriptors.
    """
    from music21 import converter

    score = converter.parse(str(midi_path))
    sf = score.flatten()

    notes = [n for n in sf.notes if getattr(n, "isNote", False)]
    chords = [c for c in sf.notes if getattr(c, "isChord", False)]

    all_pitches, all_elements = _build_pitch_sequences(sf, notes, chords)

    if not all_pitches:
        # Handle completely empty / non-pitched content
        return {
            "note_count": 0,
            "min_midi": None,
            "max_midi": None,
            "interval_entropy": 0.0,
            "step_vs_leap": 0.0,
            "direction_changes": 0,
            "repeat_rate": 0.0,
            "note_density": None,
            "avg_interval_size": 0.0,
            "rhythmic_diversity": 0,
            "bar_count_midi": 0,
            "beats_per_bar_match_rate": None,
            "detected_key": None,
            "in_key_pct_time_weighted": None,
            "ends_on_tonic": None,
            "beats_ok": None,
            "bars_ok_musical": None,
            "range_ok": None,
            "in_key_ok": None,
            "expected_bars_inferred": None,
            "tonal_stability_score": None,
            "key_changes": None,
            "drift_detected": None,
            "contour_types": [],
            "contour_diversity": 0.0,
            "ascending_tendency": 0.0,
            "descending_tendency": 0.0,
            "arch_shapes": 0,
            "downward_arches": 0,
            "config": {
                "declared_key_pc_mode": declared_key_pc_mode,
                "declared_time": declared_time,
                "range_bounds": range_bounds,
                "in_key_threshold": in_key_threshold,
            },
        }

    sequential_pitches, intervals = _compute_intervals_from_elements(all_elements)

    interval_entropy = _compute_interval_entropy(intervals)

    steps = sum(1 for d in intervals if abs(d) <= 2)
    leaps = sum(1 for d in intervals if abs(d) > 2)
    total_interval_count = steps + leaps
    step_vs_leap = (steps / total_interval_count) if total_interval_count else 0.0

    direction_changes = sum(
        1
        for i in range(1, len(intervals))
        if (intervals[i - 1] < 0 < intervals[i])
        or (intervals[i - 1] > 0 > intervals[i])
    )

    repeats = sum(
        1
        for i in range(1, len(sequential_pitches))
        if sequential_pitches[i] == sequential_pitches[i - 1]
    )
    repeat_rate = repeats / max(1, len(sequential_pitches) - 1)

    # Prefer the part with the most measures; otherwise fall back to global measures
    if score.parts:
        measure_candidates = [
            list(part.getElementsByClass("Measure"))
            for part in score.parts
        ]
        measures = max(measure_candidates, key=len) if measure_candidates else []
        if not measures:
            measures = list(score.getElementsByClass("Measure"))
    else:
        measures = list(score.getElementsByClass("Measure"))

    bar_count = len(measures)

    beats_per_bar_match_rate = _compute_beats_per_bar_match_rate(
        score, measures, declared_time, beats_epsilon
    )

    detected_key = _compute_detected_key(score)

    in_key_time_share, ends_on_tonic, in_key_ok = _compute_in_key_share_and_cadence(
        notes,
        chords,
        sequential_pitches,
        declared_key_pc_mode,
        in_key_threshold,
    )

    expected_bars_inferred = _compute_expected_bars(
        sf, declared_time, bar_count, all_pitches
    )
    bars_ok_musical = None
    if bar_count > 0 and expected_bars_inferred is not None:
        bars_ok_musical = abs(bar_count - expected_bars_inferred) <= 1

    rhythmic_diversity = _compute_rhythmic_diversity(notes, chords)

    avg_interval_size = (
        sum(abs(i) for i in intervals) / len(intervals) if intervals else 0.0
    )

    note_density = (
        len(all_pitches) / bar_count if bar_count else len(all_pitches)
    )

    beats_ok = (
        (beats_per_bar_match_rate is not None)
        and (beats_per_bar_match_rate >= 0.95)
    )

    range_ok = _compute_range_ok(all_pitches, range_bounds)

    tonal_stability = _compute_tonal_stability(score, declared_key_pc_mode)

    contour_analysis = _compute_contour_analysis(sequential_pitches)

    return {
        "note_count": len(all_pitches),
        "chord_count": len(chords),
        "element_count": len(sequential_pitches),
        "min_midi": min(all_pitches),
        "max_midi": max(all_pitches),
        "bar_count_midi": bar_count,
        "detected_key": detected_key,
        "interval_entropy": float(interval_entropy),
        "step_vs_leap": float(step_vs_leap),
        "direction_changes": int(direction_changes),
        "beats_per_bar_match_rate": (
            float(beats_per_bar_match_rate)
            if beats_per_bar_match_rate is not None else None
        ),
        "in_key_pct_time_weighted": (
            float(in_key_time_share)
            if in_key_time_share is not None else None
        ),
        "ends_on_tonic": bool(ends_on_tonic) if ends_on_tonic is not None else None,
        "beats_ok": bool(beats_ok) if beats_ok is not None else None,
        "bars_ok_musical": (
            bool(bars_ok_musical) if bars_ok_musical is not None else None
        ),
        "range_ok": bool(range_ok) if range_ok is not None else None,
        "in_key_ok": bool(in_key_ok) if in_key_ok is not None else None,
        "expected_bars_inferred": (
            int(expected_bars_inferred)
            if expected_bars_inferred is not None else None
        ),
        "repeat_rate": float(repeat_rate),
        "note_density": float(note_density) if note_density is not None else None,
        "avg_interval_size": float(avg_interval_size),
        "rhythmic_diversity": int(rhythmic_diversity),
        "tonal_stability_score": tonal_stability["tonal_stability_score"],
        "key_changes": tonal_stability["key_changes"],
        "drift_detected": tonal_stability["drift_detected"],
        "contour_types": contour_analysis["contour_types"],
        "contour_diversity": contour_analysis["contour_diversity"],
        "ascending_tendency": contour_analysis["ascending_tendency"],
        "descending_tendency": contour_analysis["descending_tendency"],
        "arch_shapes": contour_analysis["arch_shapes"],
        "downward_arches": contour_analysis["downward_arches"],
        "config": {
            "declared_key_pc_mode": declared_key_pc_mode,
            "declared_time": declared_time,
            "range_bounds": range_bounds,
            "in_key_threshold": in_key_threshold,
        },
    }


def eval_midi(
    midi_path: str | Path,
    *,
    declared_key_pc_mode: Optional[Tuple[int, str]] = None,
    declared_time: Optional[Tuple[int, int]] = None,
    range_bounds: Optional[Tuple[int, int]] = None,
    in_key_threshold: float = DEFAULT_IN_KEY_THRESHOLD,
    beats_epsilon: float = DEFAULT_BEATS_EPSILON,
) -> Dict[str, Any]:
    """
    Public entry point: evaluate a single MIDI file and wrap metrics
    with timing, error, and tooling information.
    """
    import time

    midi_path = Path(midi_path)
    start_time = time.perf_counter()

    try:
        import music21  # lazy import for clearer error reporting
        tooling = {"music21_version": getattr(music21, "__version__", None)}
    except Exception as exc:
        return {
            "ok": False,
            "seconds": 0.0,
            "metrics": None,
            "error": _truncate_err(f"music21 import failed: {exc}"),
            "tooling": {"music21_version": None},
        }

    if not midi_path.exists():
        return {
            "ok": False,
            "seconds": 0.0,
            "metrics": None,
            "error": f"MIDI not found: {midi_path}",
            "tooling": tooling,
        }

    try:
        metrics = _analyze_midi_with_music21(
            midi_path,
            declared_key_pc_mode=declared_key_pc_mode,
            declared_time=declared_time,
            range_bounds=range_bounds,
            in_key_threshold=in_key_threshold,
            beats_epsilon=beats_epsilon,
        )
        elapsed = time.perf_counter() - start_time
        return {
            "ok": True,
            "seconds": round(elapsed, 4),
            "metrics": metrics,
            "error": None,
            "tooling": tooling,
        }
    except Exception as exc:
        elapsed = time.perf_counter() - start_time
        return {
            "ok": False,
            "seconds": round(elapsed, 4),
            "metrics": None,
            "error": _truncate_err(str(exc)),
            "tooling": tooling,
        }


## Block 3 - LilyPond text-level evaluation (`eval_lily`)

This block inspects the **LilyPond source text itself** to see how well the model followed the prompt constraints, even before (or in addition to) actual rendering.

Key responsibilities:

- **Declared key/time & notation detection**
  - `_extract_declared()` parses the `.ly` file for:
    - Declared key as `(tonic_pc, mode)`
    - Declared time signature `(num, den)`
    - Notation type: `"relative"` vs `"absolute"` (based on `\relative` usage)

- **Body extraction & tokenization**
  - `strip_comments()` removes `%` line comments and `%{ ... %}` block comments.
  - `extract_music_body()` tries to isolate the main music block:
    - Prefer a `\relative c' { ... }` block.
    - Otherwise take the first `{ ... }` block or fall back to whole cleaned text.
  - `tokenize_notes_and_bars()` turns the body into a flat sequence:
    - `("note", name, accidental, marks, duration, dots)`
    - `("bar",)` and `("bar_cmd", ...)`
  - `bars_from_tokens()` groups tokens into literal bars.

- **Prompt adherence checks**
  - Notation policy:
    - If `expected_notation="relative"`, require at least one `\relative c' { ... }` anchor.
    - If `expected_notation="absolute"`, require no `\relative`.
  - Case policy:
    - When `require_lowercase=True`, disallow uppercase note names (C, D, E, …).
  - Forbidden constructs:
    - Rests (`r`), chords (`<...>`), multi-voice constructs (`<< >> \\`), repeats, tuplets, ties, grace notes, skips, `\score` and `\layout` blocks inside the body, etc.
  - Structural checks:
    - `bar_count_literal`: number of bars inferred from `|` / `\bar`.
    - `final_barline_literal_ok`: last token is a bar/bar_cmd (optional).
    - `bars_literal_ok`: bar count close (±1) to `expected_bars_literal`.
    - `notes_per_bar_literal_ok`: bar durations close to `expected_notes_per_bar_literal` in quarter-note units.
  - Accidentals policy:
    - Optionally disallow accidentals if `allow_accidentals=False`.

- **Compile / parse validation**
  - If `lilypond_bin` is provided:
    - Write a temp `.ly` (inserting `\version` if missing).
    - Invoke LilyPond with a “null” backend to just check for syntax.
    - Capture return code and a truncated error message.
  - Otherwise, or if LilyPond is not available:
    - Try a music21-based LilyPond parse as a lighter sanity check.
  - Fields:
    - `compiles` (True/False/None),
    - `compile_error`,
    - `compile_via` (`"lilypond"` / `"music21"` / None),
    - `parser_error` (for music21 failures).

- **Adherence score**
  - Collect all boolean flags (key/time present, notation_ok, lowercase_ok, no_forbidden, etc.).
  - Compute a normalized `adherence_text` in `[0,1]` plus a list of `failed_flags`.

- **Public entry**
  - `eval_lily(out_ly, ...)`:
    - Loads a `.ly` file from disk.
    - Calls `evaluate_lilypond_text()` with our configuration.
    - Wraps the result with timing and `tooling` info (`lilypond_bin` path).

This block tells us **how “clean” and on-spec the LilyPond text is**, independently of whether the MIDI rendering and musical content are good.


In [5]:
from __future__ import annotations

from pathlib import Path
from typing import Optional, Dict, Any, Tuple, List
import re
import tempfile
import subprocess
import time
import os

# Optional import: used only for the music21-based fallback parser
try:
    import music21 as _m21
except Exception:
    _m21 = None


# ---------- Regexes & key/time helpers ----------

# Regexes for detecting key and time signature in LilyPond code
KEY_RE = re.compile(r"\\key\s+([a-g])(isis|eses|is|es)?\s+\\([A-Za-z]+)")
TIME_RE = re.compile(r"\\time\s+(\d+)\s*/\s*(\d+)")

# More explicit variants for declared key/time parsing
_KEY_RE = re.compile(r"\\key\s+([a-g])(isis|eses|is|es)?\s+\\([A-Za-z]+)")
_TIME_RE = re.compile(r"\\time\s+(\d+)\s*/\s*(\d+)")

# Relative-notation detection
_REL_ANY_RE = re.compile(r"\\relative\b")
_REL_CPRIME_RE = re.compile(r"\\relative\s+c'\s*\{")


_NAME_TO_PC = {"c": 0, "d": 2, "e": 4, "f": 5, "g": 7, "a": 9, "b": 11}
_ACC_OFFSETS = {"is": +1, "isis": +2, "es": -1, "eses": -2}


def _pc_from_name_acc(letter: str, acc: str | None) -> int:
    """Convert a LilyPond pitch letter + accidental into a pitch class (0–11)."""
    return (_NAME_TO_PC[letter] + _ACC_OFFSETS.get(acc or "", 0)) % 12


def _extract_declared(ly_path: Path) -> tuple[tuple[int, str] | None, tuple[int, int] | None, str]:
    """
    Read a LilyPond file and extract:

    - Declared key as (tonic_pc, mode) or None
    - Declared time signature as (num, den) or None
    - Notation type: 'relative' or 'absolute'
    """
    txt = ly_path.read_text(encoding="utf-8", errors="ignore")

    m_key = _KEY_RE.search(txt)
    m_time = _TIME_RE.search(txt)

    key_pcm: tuple[int, str] | None = None
    if m_key:
        letter, acc, mode = m_key.groups()
        key_pcm = (_pc_from_name_acc(letter, acc), mode.lower())

    time_sig = (int(m_time.group(1)), int(m_time.group(2))) if m_time else None
    notation = "relative" if (_REL_CPRIME_RE.search(txt) or _REL_ANY_RE.search(txt)) else "absolute"

    return key_pcm, time_sig, notation


# ---------- Note / bar tokenization ----------

NOTE_RE = re.compile(r"([a-g])(isis|eses|is|es)?([',]*)(\d+)(\.*)")
BAR_RE = re.compile(r"\|")
BAR_CMD_RE = re.compile(r"\bar\s+(\"[^\"]+\"|\S+)")

REL_CPRIME_RE = re.compile(r"\\relative\s+c'\s*\{")
REL_PLAIN_C_RE = re.compile(r"\\relative\s+c\s*\{")
REL_ANY_RE = re.compile(r"\\relative\b")

OCTAVE_MARKS_IN_BODY_RE = re.compile(r"\b[a-g][',]+\d\b")
UPPERCASE_NOTE_IN_BODY_RE = re.compile(r"\b[A-G](?:isis|eses|is|es)?[',]*\d(?:\.*)?")


FORBIDDEN_PATTERNS = {
    "rests": re.compile(r"(?<!\\)\br(?:\d+(?:\.*)?)?\b"),
    "chords": re.compile(r"<\s*[a-g]"),          # allows \< and \> hairpins
    "voices": re.compile(r"<<|\\\\|>>"),
    "repeat": re.compile(r"\\repeat\b"),
    "tuplet": re.compile(r"\\tuplet\b"),
    "ties": re.compile(r"~"),
    "grace": re.compile(r"\\grace\b|\\acciaccatura\b|\\appoggiatura\b"),
    "skips": re.compile(r"\bs(?:\d+(?:\.*)?)?\b"),
    "score": re.compile(r"\\score\b"),
    "layout": re.compile(r"\\layout\b"),
}


def _truncate(s: str | None, n: int = 300) -> str | None:
    """Trim a string to at most n characters, appending an ellipsis if necessary."""
    if not s:
        return s
    s = s.strip()
    return (s[:n] + "…") if len(s) > n else s


def strip_comments(text: str) -> str:
    """Remove block (%{ ... %}) and line (%) comments from LilyPond source text."""
    text = re.sub(r'%\{[\s\S]*?%\}', '', text)
    text = re.sub(r'(?m)%.*$', '', text)
    return text


def extract_music_body(text: str) -> str:
    """
    Extract the main music body from LilyPond text.

    Priority:
    - If a \\relative c' { ... } block exists, return its content.
    - Else, return the content of the first {...} block.
    - If nothing matches, return the full comment-stripped text.
    """
    code = strip_comments(text)

    match = re.search(r"\\relative\s+c'\s*\{(.*?)\}", code, re.S)
    if not match:
        match = re.search(r"\{(.*?)\}", code, re.S)

    return (match.group(1) if match else code).strip()


def tokenize_notes_and_bars(body: str) -> List[Tuple]:
    """
    Tokenize a LilyPond body string into a flat sequence of:

    - ("note", name, acc, marks, dur, dots)
    - ("bar",)
    - ("bar_cmd", arg)
    """
    tokens: List[Tuple] = []
    i, length = 0, len(body)

    while i < length:
        note_match = NOTE_RE.match(body, i)
        if note_match:
            n, acc, marks, dur, dots = note_match.groups()
            tokens.append(
                ("note", n, acc or "", marks, int(dur), len(dots))
            )
            i = note_match.end()
            continue

        bar_cmd_match = BAR_CMD_RE.match(body, i)
        if bar_cmd_match:
            tokens.append(("bar_cmd", bar_cmd_match.group(1)))
            i = bar_cmd_match.end()
            continue

        bar_match = BAR_RE.match(body, i)
        if bar_match:
            tokens.append(("bar",))
            i = bar_match.end()
            continue

        # No token matched; advance one character
        i += 1

    return tokens


def _duration_to_quarters(length: int, dots: int) -> float:
    """Convert a LilyPond note length plus dots to quarter-note units."""
    if length <= 0:
        return 0.0

    base = 4 / float(length)
    total = base
    for i in range(1, dots + 1):
        total += base / (2 ** i)

    return total


def _bar_quarter_length(bar_tokens: List[Tuple]) -> float:
    """Sum the quarter-note length of all note tokens within a bar."""
    total = 0.0
    for tok in bar_tokens:
        if tok and tok[0] == "note":
            _, _, _, _, dur, dots = tok
            total += _duration_to_quarters(dur, dots)
    return total


def bars_from_tokens(tokens: List[Tuple]) -> List[List[Tuple]]:
    """
    Group a flat token list into bars, splitting whenever a bar/bar_cmd token appears.
    """
    bars: List[List[Tuple]] = []
    current_bar: List[Tuple] = []

    for token in tokens:
        if token[0] in {"bar", "bar_cmd"}:
            if current_bar:
                bars.append(current_bar)
                current_bar = []
        else:
            current_bar.append(token)

    if current_bar:
        bars.append(current_bar)

    return bars


# ---------- Core text evaluation (metrics only) ----------

def evaluate_lilypond_text(
    lily_text: str,
    *,
    expected_notation: str = "relative",        # "relative" or "absolute"
    require_lowercase: bool = True,
    expected_bars_literal: Optional[int] = 8,
    expected_notes_per_bar_literal: Optional[int] = 4,
    allow_accidentals: bool = True,
    require_final_barline_literal: bool = True,
    lilypond_bin: Optional[str] = None,         # if provided, try real compile; else try music21 parser
) -> Dict[str, Any]:
    """
    Evaluate LilyPond source text for structural and stylistic adherence,
    and optionally check compilability via LilyPond or music21.

    Returns a metrics dict containing:
    - Textual flags (has_key_time, notation_ok, lowercase_ok, etc.)
    - Compile/parse results (compiles, compile_error, compile_via, parser_error)
    - Overall adherence score and failed flag names
    - The evaluation configuration used
    """
    stripped = strip_comments(lily_text)
    body = extract_music_body(lily_text)

    # Key and time signature presence
    has_key = bool(KEY_RE.search(stripped))
    has_time = bool(TIME_RE.search(stripped))
    has_key_time = has_key and has_time

    # Notation policy (relative / absolute)
    rel_count = len(REL_ANY_RE.findall(stripped))
    has_rel_cprime = bool(REL_CPRIME_RE.search(stripped))
    has_rel_c_plain = bool(REL_PLAIN_C_RE.search(stripped))

    notation_mode = (expected_notation or "relative").lower()
    if notation_mode == "relative":
        notation_ok = (rel_count >= 1 and has_rel_cprime)
        relative_ok = notation_ok
    else:
        notation_ok = (rel_count == 0)
        relative_ok = False

    # Lowercase policy (disallow uppercase note names if requested)
    lowercase_ok = True
    if require_lowercase:
        lowercase_ok = not bool(UPPERCASE_NOTE_IN_BODY_RE.search(body))

    # Forbidden constructs (rests, chords, multi-voice, etc.)
    forb_hits = {
        name: bool(
            pattern.search(body if name not in {"score", "layout"} else stripped)
        )
        for name, pattern in FORBIDDEN_PATTERNS.items()
    }
    no_forbidden = not any(forb_hits.values())

    # Basic bar/token analysis
    tokens = tokenize_notes_and_bars(body)
    bars = bars_from_tokens(tokens)

    # Final barline literal check
    final_barline_literal_ok = True
    if require_final_barline_literal:
        last_bar_token = next(
            (t for t in reversed(tokens) if t and t[0] in {"bar", "bar_cmd"}),
            None,
        )
        final_barline_literal_ok = bool(last_bar_token) and bool(tokens) and (
            tokens[-1][0] in {"bar", "bar_cmd"}
        )

    # Number of literal bars (tolerate ±1 difference if configured)
    bars_literal_ok = True
    if expected_bars_literal is not None:
        bars_literal_ok = (
            abs(len(bars) - expected_bars_literal) <= 1
        ) and final_barline_literal_ok

    # Notes-per-bar approximation via quarter-note lengths
    notes_per_bar_literal_ok = True
    if expected_notes_per_bar_literal is not None and bars:
        target_quarters = float(expected_notes_per_bar_literal)
        notes_per_bar_literal_ok = all(
            abs(_bar_quarter_length(bar) - target_quarters) <= 0.25
            for bar in bars
        )

    # Accidentals policy
    accidentals_present = any(
        (token[0] == "note" and token[2])
        for token in tokens
    )
    accidentals_ok = allow_accidentals or not accidentals_present

    # Compile / parse checks (first LilyPond, then music21 fallback)
    compiles: Optional[bool] = None
    compile_error: Optional[str] = None
    compile_via: Optional[str] = None
    parser_error: Optional[str] = None

    if lilypond_bin:
        lilypond_bin_str = str(lilypond_bin)
        if not Path(lilypond_bin_str).exists():
            compiles = False
            compile_error = f"lilypond not found: {lilypond_bin_str}"
            compile_via = "lilypond"
        else:
            try:
                with tempfile.TemporaryDirectory() as td:
                    tmp_dir = Path(td)
                    test_file = tmp_dir / "test.ly"

                    src = (
                        lily_text
                        if "\\version" in lily_text
                        else '\\version "2.24.4"\n' + lily_text
                    )
                    test_file.write_text(src, encoding="utf-8")

                    result = subprocess.run(
                        [
                            lilypond_bin_str,
                            "-dno-print-pages",
                            "-dbackend=null",
                            "-o",
                            str(tmp_dir / "out"),
                            str(test_file),
                        ],
                        stdout=subprocess.PIPE,
                        stderr=subprocess.PIPE,
                        encoding="utf-8",
                        timeout=15,
                    )
                    compiles = (result.returncode == 0)
                    compile_via = "lilypond"
                    if not compiles:
                        lines = (result.stderr or result.stdout or "Compilation failed").strip().splitlines()
                        first_line = lines[0] if lines else "Compilation failed"
                        compile_error = first_line[:300]
            except subprocess.TimeoutExpired:
                compiles = False
                compile_error = "Compile timeout"
                compile_via = "lilypond"
            except Exception as e:
                compiles = False
                compile_error = str(e)[:300]
                compile_via = "lilypond"

    # Fallback: try music21 parsing if no compile result yet
    if compiles is None and _m21 is not None:
        try:
            src = (
                lily_text
                if "\\version" in lily_text
                else '\\version "2.24.4"\n' + lily_text
            )
            _m21.converter.parseData(src, format="lilypond")
            compiles = True
            compile_via = "music21"
        except Exception as e:
            compiles = False
            compile_via = "music21"
            parser_error = str(e)[:300]

    # Aggregate adherence from all boolean flags
    flags: Dict[str, bool] = {
        "has_key_time": has_key_time,
        "notation_ok": notation_ok,
        "relative_ok": relative_ok if notation_mode == "relative" else True,
        "lowercase_ok": lowercase_ok,
        "no_forbidden": no_forbidden,
        "bars_literal_ok": bars_literal_ok,
        "notes_per_bar_literal_ok": notes_per_bar_literal_ok,
        "final_barline_literal_ok": final_barline_literal_ok,
        "accidentals_ok": accidentals_ok,
    }
    if compiles is not None:
        flags["compiles"] = bool(compiles)

    adherence = (
        sum(bool(v) for v in flags.values()) / len(flags)
        if flags
        else 0.0
    )
    failed_flags = [name for name, ok in flags.items() if not ok]

    return {
        # text flags
        "has_key_time": bool(has_key_time),
        "relative_anchor_cprime": bool(has_rel_cprime),
        "relative_anchor_plain_c": bool(has_rel_c_plain),
        "notation_ok": bool(notation_ok),
        "lowercase_ok": bool(lowercase_ok),
        "no_forbidden": bool(no_forbidden),
        "forbidden_hits": forb_hits,
        "bars_literal_ok": bool(bars_literal_ok),
        "notes_per_bar_literal_ok": bool(notes_per_bar_literal_ok),
        "final_barline_literal_ok": bool(final_barline_literal_ok),
        "bar_count_literal": int(len(bars)),
        "accidentals_present": bool(accidentals_present),
        "accidentals_ok": bool(accidentals_ok),

        # compile/parse results
        "compiles": None if compiles is None else bool(compiles),
        "compile_error": compile_error,
        "compile_via": compile_via,     # "lilypond" | "music21" | None
        "parser_error": parser_error,   # only set when compile_via == "music21" and failed

        # overall adherence
        "adherence_text": float(adherence),
        "failed_flags": failed_flags,
        "config": {
            "expected_notation": expected_notation,
            "require_lowercase": require_lowercase,
            "expected_bars_literal": expected_bars_literal,
            "expected_notes_per_bar_literal": expected_notes_per_bar_literal,
            "allow_accidentals": allow_accidentals,
            "require_final_barline_literal": require_final_barline_literal,
            "lilypond_bin": str(lilypond_bin) if lilypond_bin else None,
        },
    }


# ---------- Public: single-file wrapper ----------

def eval_lily(
    out_ly: str | Path,
    *,
    expected_notation: str = "relative",
    require_lowercase: bool = True,
    expected_bars_literal: Optional[int] = 8,
    expected_notes_per_bar_literal: Optional[int] = 4,
    allow_accidentals: bool = True,
    require_final_barline_literal: bool = True,
    lilypond_bin: Optional[str | Path] = None,
) -> Dict[str, Any]:
    """
    Load a LilyPond file from disk, evaluate its text with
    `evaluate_lilypond_text`, and wrap the result with timing and tooling info.

    Returns a dict:
    {
        "ok": bool,
        "seconds": float,
        "metrics": dict | None,
        "error": str | None,
        "tooling": { "lilypond_bin": str | None },
    }
    """
    start = time.perf_counter()
    out_ly = Path(out_ly)

    tooling = {
        "lilypond_bin": str(lilypond_bin) if lilypond_bin else None,
    }

    if not out_ly.exists():
        return {
            "ok": False,
            "seconds": 0.0,
            "metrics": None,
            "error": f"LilyPond file not found: {out_ly}",
            "tooling": tooling,
        }

    try:
        text = out_ly.read_text(encoding="utf-8")
    except Exception as e:
        return {
            "ok": False,
            "seconds": 0.0,
            "metrics": None,
            "error": _truncate(str(e)),
            "tooling": tooling,
        }

    try:
        metrics = evaluate_lilypond_text(
            text,
            expected_notation=expected_notation,
            require_lowercase=require_lowercase,
            expected_bars_literal=expected_bars_literal,
            expected_notes_per_bar_literal=expected_notes_per_bar_literal,
            allow_accidentals=allow_accidentals,
            require_final_barline_literal=require_final_barline_literal,
            lilypond_bin=str(lilypond_bin) if lilypond_bin else None,
        )
        dt = time.perf_counter() - start
        return {
            "ok": True,
            "seconds": round(dt, 4),
            "metrics": metrics,
            "error": None,
            "tooling": tooling,
        }
    except Exception as e:
        dt = time.perf_counter() - start
        return {
            "ok": False,
            "seconds": round(dt, 4),
            "metrics": None,
            "error": _truncate(str(e)),
            "tooling": tooling,
        }


## Block 4 - Integrated pipeline runner & batch summaries

This final block stitches everything together into a **single end-to-end pipeline** operating on the `zero_shot_outputs` directory.

Main responsibilities:

- **Paths & environment**
  - `_nb_base_dir()` and `OUTPUTS_ROOT` define where to look for:
    - `raw/` → model generations (`out_*.ly`)
    - `midi/` → rendered MIDI and logs
    - `eval_midi/` → MIDI evaluation JSONL
    - `eval_lily/` → text evaluation JSONL
    - `final/` → combined per-run + per-batch summaries
  - `find_lilypond_bin()` tries to auto-detect a LilyPond binary and reports it in a small Markdown banner.

- **Batch discovery & bookkeeping**
  - `_collect_batches()` groups `out_*.ly` files by batch subdirectory.
  - `list_batches()` lists available batch names.
  - `_load_generation_map()` loads `summary.json` from generation-time and maps `run` indices to generation metadata (tokens, elapsed seconds, model, etc.).

- **Empty-output detection**
  - `_is_effectively_empty_ly()` and helpers detect “empty” generations:
    - Only whitespace/comments.
    - No notes, blocks, or key/time directives.
  - These are flagged upfront as `status="empty"` and skip heavy stages.

- **Status logic**
  - `_status_from(ok_midi_render, ok_midi_eval, ok_lily_eval, is_empty, compiles)` collapses all stage outcomes into a single label:
    - `"empty"`
    - `"failed_to_compile"` (LilyPond text doesn’t compile/parse)
    - `"ok"` (render, MIDI eval, and Lily eval all ok)
    - `"lily_only_ok"` (Lily eval ok but MIDI render/eval failed)
    - `"failed"` (other combinations)

- **Per-file pipeline**
  For each `out_*.ly` in a batch:
  1. Detect `is_empty` and try to extract declared key/time/notation with `_extract_declared()`.
  2. Run **Lily → MIDI** via `lily_to_midi()` (unless empty).
  3. Run **MIDI evaluation** via `eval_midi()` if MIDI exists.
  4. Run **Lily text evaluation** via `eval_lily()` (unless empty).
  5. Consolidate everything into a single `final_rec` with:
     - generation metadata,
     - render + MIDI metrics,
     - Lily metrics,
     - `status` and a `style_label` (`"scalar"` vs `"balanced"` based on step/entropy heuristics).
  6. Append that record to `final/index.jsonl`.

- **Human-readable tables**
  - Build a per-batch DataFrame with key columns (compile status, musical metrics, adherence, style label, note counts, etc.).
  - Convert booleans into emojis (✅/❌) where appropriate.
  - Round numeric values for display (3 decimals).
  - Special formatting:
    - `in_key_pct_time_weighted` and `adherence_text` are shown as percentages if they look like fractions.

- **AVG row & JSON summary**
  - Compute an extra **AVG row** per batch:
    - Percent “ok”.
    - Compile / in-key / beats / bars / drift rates.
    - Mean in-key percentage, beats-per-bar match rate, tonal stability, contour diversity, ascending tendency.
    - Mean adherence, plus “% scalar” among compiled rows.
    - Average note count and literal bar count over successful runs.
  - Attach this AVG row to the bottom of the displayed table.
  - Serialize it to `final/AVG_row.json` for downstream analysis (plots, thesis tables).

- **Public API & simple UI**
  - `run_full_pipeline_for_all(...)` is the main programmatic entry point; it returns:
    - `dfs_by_batch`: the displayed DataFrames (including AVG row)
    - `avg_rows_by_batch`: plain dict summaries.

This block turns the three previously defined components-renderer, MIDI evaluator, and Lily text evaluator-into a **repeatable, dataset-wide evaluation pipeline** with structured logs and ready-to-use aggregate metrics.


In [6]:
from __future__ import annotations

from pathlib import Path
from typing import Optional, Dict, Any, Tuple, List

import json
import re
import time
import hashlib
import os
import sys
import shutil

import pandas as pd
from IPython.display import display, Markdown


# ---------- Paths & environment ----------

def _nb_base_dir() -> Path:
    """Return the base directory for this notebook/module, or CWD as fallback."""
    try:
        return Path(__file__).resolve().parent
    except NameError:
        return Path.cwd().resolve()


OUTPUTS_ROOT = _nb_base_dir() / "zero_shot_outputs"


def find_lilypond_bin() -> str | None:
    """
    Try to locate a usable LilyPond executable.

    Search order:
      1. LILYPOND_BIN environment variable
      2. PATH lookup
      3. A small set of common Windows install paths
    """
    env_value = os.environ.get("LILYPOND_BIN")
    if env_value and Path(env_value).exists():
        return str(Path(env_value))

    candidate = shutil.which("lilypond.exe" if sys.platform.startswith("win") else "lilypond")
    if candidate:
        return candidate

    windows_candidates = [
        r"C:\lilypond-2.24.4-mingw-x86_64\lilypond-2.24.4\bin\lilypond.exe",
        r"C:\Program Files (x86)\LilyPond\usr\bin\lilypond.exe",
        r"C:\Program Files\LilyPond\usr\bin\lilypond.exe",
    ]
    for path in windows_candidates:
        if Path(path).exists():
            return path

    return None


def _rel_to_outputs(path: Path | None) -> str | None:
    """
    Represent a path relative to OUTPUTS_ROOT when possible.

    If the path cannot be relativized, return its absolute string form.
    """
    if path is None:
        return None
    try:
        return str(path.resolve().relative_to(OUTPUTS_ROOT.resolve()))
    except Exception:
        return str(path)


def _resolve_from_summary_path(paths_obj: Dict[str, Any] | None, key: str) -> Path | None:
    """
    Resolve a stored path from a summary.json record.

    Handles both absolute and relative paths and returns a Path or None.
    """
    if not paths_obj:
        return None

    raw = paths_obj.get(key)
    if not raw:
        return None

    path = Path(raw)
    if path.is_absolute():
        return path

    return (OUTPUTS_ROOT / path).resolve()


# ---------- Musical defaults ----------

DEFAULT_DECLARED_TIME: Optional[Tuple[int, int]] = (4, 4)
RANGE_BOUNDS: Optional[Tuple[int, int]] = None
IN_KEY_THRESHOLD: float = 0.98
BEATS_EPS: float = 1e-6

LILYPOND_BIN_FOR_COMPILE: Optional[str] = None
if LILYPOND_BIN_FOR_COMPILE is None:
    LILYPOND_BIN_FOR_COMPILE = find_lilypond_bin()
    display(
        Markdown(
            f"**LilyPond compiler:** `{LILYPOND_BIN_FOR_COMPILE}`"
            if LILYPOND_BIN_FOR_COMPILE
            else "**LilyPond compiler:** _not found_ (Block 3 will use music21 fallback if available)"
        )
    )


# ---------- Pandas display preferences ----------

pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
pd.set_option("display.max_colwidth", None)


# ---------- Generic helpers ----------

def _sha256_file(path: Path) -> str:
    """Compute the SHA-256 checksum of a file on disk."""
    hasher = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            hasher.update(chunk)
    return hasher.hexdigest()


def _write_jsonl_line(path: Path, record: Dict[str, Any]) -> None:
    """Append a single JSON object as one line to a JSONL file."""
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")


def _load_generation_map(batch_dir: Path) -> Dict[int, Dict[str, Any]]:
    """
    Load generation metadata from batch_dir/summary.json.

    Returns:
        Mapping run_index -> {tokens_used, elapsed_seconds, model, ...}
    """
    summary_path = batch_dir / "summary.json"
    by_run: Dict[int, Dict[str, Any]] = {}

    if not summary_path.exists():
        return by_run

    try:
        data = json.loads(summary_path.read_text(encoding="utf-8"))
    except Exception:
        return by_run

    for rec in data:
        try:
            run_index = int(rec.get("run"))
        except Exception:
            continue

        out_path = _resolve_from_summary_path(rec.get("paths"), "out_ly") if rec.get("paths") else None
        sha = _sha256_file(out_path) if (out_path and out_path.exists()) else None

        by_run[run_index] = {
            "tokens_used": rec.get("tokens_used"),
            "elapsed_seconds": rec.get("elapsed_seconds"),
            "model": rec.get("model"),
            "temperature": rec.get("temperature"),
            "prompt_file": rec.get("prompt_file"),
            "gen_sha256_out_ly": sha,
            "paths": {"out_ly": _rel_to_outputs(out_path) if out_path else None},
        }

    return by_run


def _infer_run_number(out_ly: Path) -> Optional[int]:
    """Extract the numeric run index from filenames like 'out_0001.ly'."""
    match = re.search(r"out_(\d+)\.ly$", out_ly.name)
    return int(match.group(1)) if match else None


# ---------- Empty LilyPond detection ----------

_BLOCK_COMMENT_RE = re.compile(r'%\{.*?%\}', flags=re.S)
_LINE_COMMENT_RE = re.compile(r'%.*?$', flags=re.M)
_NOTE_TOKEN_RE = re.compile(r'\b[abcdefg](?:[,\'/]*)\d\b')
_HAS_BLOCK_RE = re.compile(r'\{[^}]*\}')


def _strip_comments_and_ws(text: str) -> str:
    """Remove comments, version lines and compress whitespace for quick emptiness checks."""
    if not text:
        return ""

    text = _BLOCK_COMMENT_RE.sub("", text)
    text = _LINE_COMMENT_RE.sub("", text)
    text = re.sub(r'\\version\s+"[^"]*"\s*', "", text)
    text = re.sub(r'\s+', " ", text).strip()
    return text


def _is_effectively_empty_ly(path: Path) -> bool:
    """
    Heuristically detect "empty" LilyPond outputs with no meaningful content.

    Considers notes, blocks and key/time/relative directives.
    """
    if not path.exists():
        return True

    try:
        raw = path.read_text(encoding="utf-8", errors="ignore")
    except Exception:
        return True

    stripped = _strip_comments_and_ws(raw)
    if not stripped or len(stripped) < 20:
        return True

    has_notes = bool(_NOTE_TOKEN_RE.search(stripped))
    has_block = bool(_HAS_BLOCK_RE.search(stripped))
    has_meta = any(x in stripped for x in ("\\relative", "\\time", "\\key"))

    if not (has_notes or has_block or has_meta):
        return True

    return False


# ---------- Status & batch discovery ----------

def _status_from(
    ok_midi_render: bool,
    ok_midi_eval: bool,
    ok_lily_eval: bool,
    *,
    is_empty: bool = False,
    compiles: bool | None = None,
) -> str:
    """
    Compute a single status label for a generation/evaluation pipeline run.

    Possible statuses:
      - "empty"
      - "failed_to_compile"
      - "ok"
      - "lily_only_ok"
      - "failed"
    """
    if is_empty:
        return "empty"
    if compiles is False:
        return "failed_to_compile"
    if ok_midi_render and ok_midi_eval and ok_lily_eval:
        return "ok"
    if ok_lily_eval and not (ok_midi_render and ok_midi_eval):
        return "lily_only_ok"
    return "failed"


def _collect_batches(raw_root: Path) -> Dict[str, List[Path]]:
    """
    Walk the raw output directory and group 'out_*.ly' files by batch name.

    Returns:
        {batch_name -> list of .ly paths}
    """
    out_map: Dict[str, List[Path]] = {}
    for out_ly in raw_root.rglob("out_*.ly"):
        try:
            relative = out_ly.relative_to(raw_root)
        except Exception:
            continue
        batch_name = relative.parts[0] if relative.parts else "_root"
        out_map.setdefault(batch_name, []).append(out_ly)
    return out_map


def list_batches(outputs_root: Path = OUTPUTS_ROOT) -> List[str]:
    """List available batch subdirectories under outputs_root/raw."""
    raw_root = outputs_root / "raw"
    if not raw_root.exists():
        return []
    return sorted([p.name for p in raw_root.iterdir() if p.is_dir()])


def _summarize(records: List[Dict[str, Any]]) -> Dict[str, Any]:
    """
    Build a compact summary dict from a list of final_records in index.jsonl.

    Includes basic counts, status rates, token totals and average stage timings.
    """
    count = len(records)
    if count == 0:
        return {"count": 0}

    statuses = [r.get("status") for r in records]
    ok_rate = sum(1 for s in statuses if s == "ok") / count
    lily_ok_rate = sum(1 for s in statuses if s in ("ok", "lily_only_ok")) / count

    token_values = [((r.get("gen") or {}).get("tokens_used")) for r in records]
    token_values = [t for t in token_values if isinstance(t, (int, float))]

    gen_secs = [((r.get("gen") or {}).get("elapsed_seconds")) for r in records]
    gen_secs = [t for t in gen_secs if isinstance(t, (int, float))]

    render_secs = [
        ((r.get("midi_render") or {}).get("seconds"))
        for r in records
        if (r.get("midi_render") or {}).get("seconds") is not None
    ]
    midi_eval_secs = [
        ((r.get("midi_eval") or {}).get("seconds"))
        for r in records
        if (r.get("midi_eval") or {}).get("seconds") is not None
    ]
    lily_eval_secs = [
        ((r.get("lily_eval") or {}).get("seconds"))
        for r in records
        if (r.get("lily_eval") or {}).get("seconds") is not None
    ]

    return {
        "count": count,
        "status_rates": {
            "ok": ok_rate,
            "lily_or_ok": lily_ok_rate,
            "failed": sum(1 for s in statuses if s == "failed") / count,
            "empty": sum(1 for s in statuses if s == "empty") / count,
        },
        "tokens": {
            "total": sum(token_values) if token_values else None,
            "avg": (sum(token_values) / len(token_values)) if token_values else None,
        },
        "gen_time_seconds": {
            "total": sum(gen_secs) if gen_secs else None,
            "avg": (sum(gen_secs) / len(gen_secs)) if gen_secs else None,
        },
        "stage_times_seconds_avg": {
            "midi_render": (sum(render_secs) / len(render_secs)) if render_secs else None,
            "eval_midi": (sum(midi_eval_secs) / len(midi_eval_secs)) if midi_eval_secs else None,
            "eval_lily": (sum(lily_eval_secs) / len(lily_eval_secs)) if lily_eval_secs else None,
        },
    }


# ---------- Stats helpers ----------

def _rate_over_total(series_like, missing_as=False) -> float | None:
    """
    Compute the fraction of True values over total entries in a series.

    missing_as controls how None / unknown values are treated:
      - False: count as False
      - True:  count as True
      - None:  discarded from the denominator
    """
    series = pd.Series(series_like, dtype="object")

    def normalize(value):
        if isinstance(value, str):
            lower = value.strip().lower()
            if lower in ("? yes", "yes", "true", "1"):
                return True
            if lower in ("? no", "no", "false", "0"):
                return False
            return None
        if isinstance(value, bool):
            return value
        return None

    series = series.map(normalize)

    if len(series) == 0:
        return None

    if missing_as is None:
        series = series.dropna()
    else:
        series = series.fillna(bool(missing_as))

    if len(series) == 0:
        return None

    return float(series.mean())


def _mean_over_total(series_like, fill_value: float = 0.0) -> float | None:
    """Compute the mean including all rows, filling NaNs with fill_value."""
    series = pd.to_numeric(series_like, errors="coerce")
    if len(series) == 0:
        return None
    return float(series.fillna(fill_value).mean())


def _mean_dropna(series_like) -> float | None:
    """Compute the mean over non-missing numeric entries only."""
    series = pd.to_numeric(series_like, errors="coerce").dropna()
    if len(series) == 0:
        return None
    return float(series.mean())


def _mean_successful_only(series_like, success_mask: pd.Series | None) -> float | None:
    """
    Average a metric over successful (status == 'ok') rows only.

    Drops missing values and zeros before averaging.
    """
    if success_mask is None or not success_mask.any():
        return None

    series = pd.to_numeric(series_like, errors="coerce")
    series_success = series[success_mask].dropna()
    series_success = series_success[series_success > 0]

    if len(series_success) == 0:
        return None

    return float(series_success.mean())


# ---------- Time-signature helpers ----------

def _expected_notes_per_bar_from_time(
    time_sig: Optional[Tuple[int, int]],
    default: Optional[float] = 4.0,
) -> Optional[float]:
    """
    Compute the expected 'quarter-note units' per bar from a time signature.

    Examples:
      - 4/4 -> 4 * (4/4) = 4.0
      - 3/4 -> 3 * (4/4) = 3.0
      - 6/8 -> 6 * (4/8) = 3.0

    If time_sig is None or invalid, fall back to `default`.
    """
    if not time_sig:
        return default

    try:
        num, den = int(time_sig[0]), int(time_sig[1])
        if den == 0:
            return default
    except Exception:
        return default

    # Convert to "quarter-note equivalents"
    return num * (4.0 / den)


# ---------- Formatting helpers for display ----------

def _fmt3_optional(value):
    """
    Format numeric values as floats with 3 decimals if possible.

    Returns an empty string for None/NaN and falls back to str(value) otherwise.
    """
    if value is None:
        return ""
    try:
        if pd.isna(value):
            return ""
    except Exception:
        pass

    try:
        return f"{float(value):.3f}"
    except Exception:
        return str(value)


def _fmt_int_optional(value):
    """
    Format numeric values as integers, hiding None/NaN as empty strings.
    """
    if value is None:
        return ""
    try:
        if pd.isna(value):
            return ""
    except Exception:
        pass

    try:
        return f"{int(value)}"
    except Exception:
        return str(value)


# ---------- Main pipeline runner ----------

def run_full_pipeline_for_all(
    *,
    outputs_root: Path = OUTPUTS_ROOT,
    only_batches: Optional[List[str]] = None,
    beats_epsilon: float = BEATS_EPS,
    in_key_threshold: float = IN_KEY_THRESHOLD,
    range_bounds=RANGE_BOUNDS,
    lily_compile_bin: Optional[str] = LILYPOND_BIN_FOR_COMPILE,
    force_midi: bool = False,
) -> tuple[Dict[str, pd.DataFrame], Dict[str, Dict]]:
    """
    Run the full end-to-end pipeline for all (or selected) batches.

    Steps for each .ly file:
      1. LilyPond → MIDI
      2. MIDI evaluation
      3. LilyPond text evaluation
      4. Aggregate metrics into per-batch tables and AVG rows

    Returns:
        (dfs_by_batch, avg_rows_by_batch)
        - dfs_by_batch: batch -> DataFrame including AVG row
        - avg_rows_by_batch: batch -> AVG row as plain dict
    """
    raw_root = outputs_root / "raw"
    midi_root = outputs_root / "midi"
    eval_midi_root = outputs_root / "eval_midi"
    eval_lily_root = outputs_root / "eval_lily"
    final_root = outputs_root / "final"

    if not raw_root.exists():
        raise FileNotFoundError(f"Raw root not found: {raw_root}")

    batch_map = _collect_batches(raw_root)
    if only_batches:
        allowed = set(only_batches)
        batch_map = {batch: files for batch, files in batch_map.items() if batch in allowed}

    dfs_by_batch: Dict[str, pd.DataFrame] = {}
    avg_rows_by_batch: Dict[str, Dict] = {}

    for batch, out_files in sorted(batch_map.items()):
        gen_map = _load_generation_map(raw_root / batch)

        midi_jsonl = midi_root / batch / "index.jsonl"
        eval_midi_jsonl = eval_midi_root / batch / "index.jsonl"
        eval_lily_jsonl = eval_lily_root / batch / "index.jsonl"
        final_jsonl = final_root / batch / "index.jsonl"

        for jsonl_path in (midi_jsonl, eval_midi_jsonl, eval_lily_jsonl, final_jsonl):
            jsonl_path.parent.mkdir(parents=True, exist_ok=True)
            jsonl_path.write_text("", encoding="utf-8")

        display_rows: List[Dict[str, Any]] = []

        # ---------- Per-file pipeline ----------
        for out_ly in sorted(out_files):
            run = _infer_run_number(out_ly) or -1
            is_empty = _is_effectively_empty_ly(out_ly)

            # Try to extract declared key / time / notation, but don't fail if it breaks
            decl_key, decl_time, notation_mode = (None, None, "absolute")
            try:
                decl_key, decl_time, notation_mode = _extract_declared(out_ly)
            except Exception:
                pass

            if not decl_time and DEFAULT_DECLARED_TIME:
                decl_time = DEFAULT_DECLARED_TIME

            # Compute expected notes per bar from time signature
            expected_notes_per_bar = _expected_notes_per_bar_from_time(
                decl_time,
                default=4.0,  # preserves old behavior when we can't infer anything
            )

            # 1) Lily → MIDI
            if is_empty:
                r_render = {
                    "ok": False,
                    "seconds": 0.0,
                    "reason": "empty_response",
                    "error": "Empty or no musical content",
                    "tooling": {},
                    "paths": {"midi": None},
                }
                midi_path: Optional[Path] = None
            else:
                midi_dir = midi_root / batch
                r_render = lily_to_midi(out_ly, midi_dir=midi_dir, force=force_midi)
                midi_path = (
                    Path(r_render["paths"]["midi"])
                    if (r_render.get("paths") and r_render["paths"].get("midi"))
                    else None
                )

            _write_jsonl_line(
                midi_jsonl,
                {
                    "run": run,
                    "paths": {
                        "out_ly": _rel_to_outputs(out_ly),
                        "midi": _rel_to_outputs(midi_path) if midi_path else None,
                    },
                    "ok": bool(r_render.get("ok")),
                    "seconds": r_render.get("seconds"),
                    "reason": r_render.get("reason"),
                    "error": r_render.get("error"),
                    "tooling": r_render.get("tooling"),
                    "created_at": time.strftime("%Y-%m-%dT%H:%M:%S"),
                },
            )

            # 2) Eval MIDI
            if (not is_empty) and r_render.get("ok") and midi_path and midi_path.exists():
                r_midi_eval = eval_midi(
                    midi_path,
                    declared_key_pc_mode=decl_key,
                    declared_time=decl_time,
                    range_bounds=range_bounds,
                    in_key_threshold=in_key_threshold,
                    beats_epsilon=beats_epsilon,
                )
            else:
                r_midi_eval = {
                    "ok": False,
                    "seconds": 0.0,
                    "metrics": None,
                    "error": ("Empty response" if is_empty else "No MIDI to evaluate"),
                    "tooling": {},
                }

            _write_jsonl_line(
                eval_midi_jsonl,
                {
                    "run": run,
                    "paths": {
                        "midi": _rel_to_outputs(midi_path) if midi_path else None,
                    },
                    "ok": bool(r_midi_eval.get("ok")),
                    "seconds": r_midi_eval.get("seconds"),
                    "metrics": r_midi_eval.get("metrics"),
                    "error": r_midi_eval.get("error"),
                    "tooling": r_midi_eval.get("tooling"),
                    "created_at": time.strftime("%Y-%m-%dT%H:%M:%S"),
                },
            )

            # 3) Eval Lily
            if is_empty:
                r_lily_eval = {
                    "ok": False,
                    "seconds": 0.0,
                    "metrics": None,
                    "error": "Empty response",
                    "tooling": {},
                }
            else:
                r_lily_eval = eval_lily(
                    out_ly,
                    expected_notation=notation_mode,
                    require_lowercase=True,
                    expected_bars_literal=8,
                    expected_notes_per_bar_literal=expected_notes_per_bar,
                    allow_accidentals=True,
                    require_final_barline_literal=True,
                    lilypond_bin=lily_compile_bin,
                )

            _write_jsonl_line(
                eval_lily_jsonl,
                {
                    "run": run,
                    "paths": {"out_ly": _rel_to_outputs(out_ly)},
                    "ok": bool(r_lily_eval.get("ok")),
                    "seconds": r_lily_eval.get("seconds"),
                    "metrics": r_lily_eval.get("metrics"),
                    "error": r_lily_eval.get("error"),
                    "tooling": r_lily_eval.get("tooling"),
                    "created_at": time.strftime("%Y-%m-%dT%H:%M:%S"),
                },
            )

            # ---------- Consolidate final record ----------
            gen_rec = gen_map.get(run, {})
            gen_sha = gen_rec.get("gen_sha256_out_ly") or (_sha256_file(out_ly) if out_ly.exists() else None)

            lily_metrics = (r_lily_eval.get("metrics") or {})
            lily_compiles = lily_metrics.get("compiles")
            midi_metrics = (r_midi_eval.get("metrics") or {})

            # If LilyPond parsing/compile failed, drop all detailed metrics
            if lily_compiles is False:
                lily_metrics = {"compiles": False}
                midi_metrics = {}

            status = _status_from(
                bool(r_render.get("ok")),
                bool(r_midi_eval.get("ok")),
                bool(r_lily_eval.get("ok")),
                is_empty=is_empty,
                compiles=lily_compiles,
            )

            # Label style only when we have compiled Lily and MIDI metrics
            if (lily_compiles is True) and midi_metrics:
                is_scalar = (
                    (
                        midi_metrics.get("step_share") is not None
                        and midi_metrics.get("step_share") >= 0.95
                    )
                    or (
                        midi_metrics.get("step_vs_leap") is not None
                        and midi_metrics.get("step_vs_leap") >= 0.95
                    )
                ) and (
                    midi_metrics.get("interval_entropy") is not None
                    and midi_metrics.get("interval_entropy") <= 1.0
                )
                style_label = "scalar" if is_scalar else "balanced"
            else:
                style_label = None

            final_rec = {
                "run": run,
                "paths": {
                    "out_ly": _rel_to_outputs(out_ly),
                    "midi": _rel_to_outputs(midi_path) if midi_path else None,
                },
                "sha256": {"out_ly": gen_sha},
                "gen": {
                    "tokens_used": gen_rec.get("tokens_used"),
                    "elapsed_seconds": gen_rec.get("elapsed_seconds"),
                    "model": gen_rec.get("model"),
                    "temperature": gen_rec.get("temperature"),
                    "prompt_file": gen_rec.get("prompt_file"),
                },
                "midi_render": {
                    "ok": bool(r_render.get("ok")),
                    "seconds": r_render.get("seconds"),
                    "reason": r_render.get("reason"),
                    "error": r_render.get("error"),
                },
                "midi_eval": {
                    "ok": bool(r_midi_eval.get("ok")),
                    "seconds": r_midi_eval.get("seconds"),
                    "metrics": midi_metrics if midi_metrics else None,
                    "error": r_midi_eval.get("error"),
                },
                "lily_eval": {
                    "ok": bool(r_lily_eval.get("ok")),
                    "seconds": r_lily_eval.get("seconds"),
                    "metrics": lily_metrics if lily_metrics else None,
                    "error": r_lily_eval.get("error"),
                },
                "status": status,
                "created_at": time.strftime("%Y-%m-%dT%H:%M:%S"),
            }
            _write_jsonl_line(final_jsonl, final_rec)

            # ---------- Row for the human-readable table ----------
            note_count_val = None
            bar_count_literal_val = None

            if status == "ok":
                note_count_val = midi_metrics.get("note_count") if midi_metrics else None
                bar_count_literal_val = lily_metrics.get("bar_count_literal") if lily_metrics else None
                if note_count_val == 0:
                    note_count_val = None
                if bar_count_literal_val == 0:
                    bar_count_literal_val = None

            display_rows.append(
                {
                    "batch": batch,
                    "run": run,
                    "compiles": lily_metrics.get("compiles") if not is_empty else False,
                    "status": status,
                    "tokens": gen_rec.get("tokens_used"),
                    "gen_s": gen_rec.get("elapsed_seconds"),
                    "bars_ok_musical": midi_metrics.get("bars_ok_musical"),
                    "beats_ok": midi_metrics.get("beats_ok"),
                    "beats_per_bar_match_rate": midi_metrics.get("beats_per_bar_match_rate"),
                    "interval_entropy": midi_metrics.get("interval_entropy"),
                    "step_vs_leap": midi_metrics.get("step_vs_leap"),
                    "step_share": (
                        midi_metrics.get("step_share")
                        if "step_share" in midi_metrics
                        else midi_metrics.get("step_vs_leap")
                    ),
                    "avg_interval_size": midi_metrics.get("avg_interval_size"),
                    "direction_changes": midi_metrics.get("direction_changes"),
                    "repeat_rate": midi_metrics.get("repeat_rate"),
                    "note_density": midi_metrics.get("note_density"),
                    "in_key_ok": midi_metrics.get("in_key_ok"),
                    "in_key_pct_time_weighted": midi_metrics.get("in_key_pct_time_weighted"),
                    "tonal_stability_score": midi_metrics.get("tonal_stability_score"),
                    "drift_detected": midi_metrics.get("drift_detected"),
                    "contour_diversity": midi_metrics.get("contour_diversity"),
                    "ascending_tendency": midi_metrics.get("ascending_tendency"),
                    "style_label": style_label,
                    "adherence_text": lily_metrics.get("adherence_text"),
                    "note_count": note_count_val,
                    "bar_count_literal": bar_count_literal_val,
                }
            )

        # ---------- Per-batch aggregation & display ----------

        final_lines = (final_root / batch / "index.jsonl").read_text(encoding="utf-8").splitlines()
        final_records = [json.loads(line) for line in final_lines if line.strip()]

        (final_root / batch / "final_summary.json").write_text(
            json.dumps(_summarize(final_records), indent=2, ensure_ascii=False),
            encoding="utf-8",
        )

        df_b = pd.DataFrame(display_rows).sort_values("run").reset_index(drop=True)

        COLUMNS = [
            "batch",
            "run",
            "compiles",
            "status",
            "tokens",
            "gen_s",
            "bars_ok_musical",
            "beats_ok",
            "beats_per_bar_match_rate",
            "interval_entropy",
            "step_share",
            "step_vs_leap",
            "avg_interval_size",
            "direction_changes",
            "repeat_rate",
            "note_density",
            "in_key_ok",
            "in_key_pct_time_weighted",
            "tonal_stability_score",
            "drift_detected",
            "contour_diversity",
            "ascending_tendency",
            "style_label",
            "adherence_text",
            "note_count",
            "bar_count_literal",
        ]
        df_b = df_b[[c for c in COLUMNS if c in df_b.columns]]

        # Raw series used for AVG calculations
        raw_compiles = df_b["compiles"].copy() if "compiles" in df_b.columns else None
        raw_in_key_pct = (
            pd.to_numeric(df_b["in_key_pct_time_weighted"], errors="coerce")
            if "in_key_pct_time_weighted" in df_b.columns
            else None
        )
        raw_beats_ok = df_b["beats_ok"].copy() if "beats_ok" in df_b.columns else None
        raw_bars_ok = df_b["bars_ok_musical"].copy() if "bars_ok_musical" in df_b.columns else None
        raw_adherence_text = (
            pd.to_numeric(df_b["adherence_text"], errors="coerce")
            if "adherence_text" in df_b.columns
            else None
        )
        raw_beats_per_bar_match_rate = (
            pd.to_numeric(df_b["beats_per_bar_match_rate"], errors="coerce")
            if "beats_per_bar_match_rate" in df_b.columns
            else None
        )
        raw_tonal_stability = (
            pd.to_numeric(df_b["tonal_stability_score"], errors="coerce")
            if "tonal_stability_score" in df_b.columns
            else None
        )
        raw_contour_diversity = (
            pd.to_numeric(df_b["contour_diversity"], errors="coerce")
            if "contour_diversity" in df_b.columns
            else None
        )
        raw_ascending_tendency = (
            pd.to_numeric(df_b["ascending_tendency"], errors="coerce")
            if "ascending_tendency" in df_b.columns
            else None
        )

        # Copy for display styling
        df_disp = df_b.copy()

        if "compiles" in df_disp.columns:
            df_disp["compiles"] = df_disp["compiles"].map(
                lambda x: "✅ Yes" if x is True else ("❌ No" if x is False else None)
            )

        int_cols = ["tokens", "note_count", "bar_count_literal"]
        for col in int_cols:
            if col in df_disp.columns:
                df_disp[col] = pd.to_numeric(df_disp[col], errors="coerce").astype("Int64")

        round3_cols = [
            "gen_s",
            "interval_entropy",
            "note_density",
            "avg_interval_size",
            "repeat_rate",
            "beats_per_bar_match_rate",
            "step_share",
            "step_vs_leap",
            "tonal_stability_score",
            "contour_diversity",
            "ascending_tendency",
        ]
        for col in round3_cols:
            if col in df_disp.columns:
                df_disp[col] = pd.to_numeric(df_disp[col], errors="coerce").round(3)

        if "in_key_pct_time_weighted" in df_disp.columns:
            df_disp["in_key_pct_time_weighted"] = pd.to_numeric(
                df_disp["in_key_pct_time_weighted"],
                errors="coerce",
            ).apply(lambda v: f"{round(100 * v, 1)}%" if pd.notna(v) else None)

        if raw_adherence_text is not None:
            non_na = raw_adherence_text.dropna()
            if len(non_na) and non_na.ge(0).all() and non_na.le(1).all():
                df_disp["adherence_text"] = raw_adherence_text.apply(
                    lambda v: f"{round(100 * v, 1)}%" if pd.notna(v) else None
                )
            else:
                df_disp["adherence_text"] = raw_adherence_text.round(3)

        # ---------- Build AVG row ----------

        n_runs = int(len(df_b))
        success_mask = df_b["status"] == "ok"

        bool_total = ["compiles", "in_key_ok", "beats_ok", "bars_ok_musical", "drift_detected"]
        pct_total = [
            "in_key_pct_time_weighted",
            "beats_per_bar_match_rate",
            "adherence_text",
            "tonal_stability_score",
            "contour_diversity",
            "ascending_tendency",
        ]
        num_dropna = [
            "tokens",
            "gen_s",
            "interval_entropy",
            "step_share",
            "step_vs_leap",
            "avg_interval_size",
            "direction_changes",
            "repeat_rate",
            "note_density",
        ]

        ok_pct = (
            float(((df_b["status"] == "ok").mean() * 100.0))
            if "status" in df_b.columns and len(df_b)
            else None
        )

        compiled_rate = _rate_over_total(raw_compiles, missing_as=False) if raw_compiles is not None else None
        in_key_rate = (
            _rate_over_total(df_b["in_key_ok"], missing_as=False)
            if "in_key_ok" in df_b.columns
            else None
        )
        beats_ok_rate = _rate_over_total(raw_beats_ok, missing_as=False) if raw_beats_ok is not None else None
        bars_ok_rate = _rate_over_total(raw_bars_ok, missing_as=False) if raw_bars_ok is not None else None
        drift_detected_rate = (
            _rate_over_total(df_b["drift_detected"], missing_as=False)
            if "drift_detected" in df_b.columns
            else None
        )

        in_key_pct_time_weighted_avg = (
            _mean_over_total(raw_in_key_pct, fill_value=0.0) if raw_in_key_pct is not None else None
        )
        beats_per_bar_match_rate_avg = (
            _mean_over_total(raw_beats_per_bar_match_rate, fill_value=0.0)
            if raw_beats_per_bar_match_rate is not None
            else None
        )
        tonal_stability_avg = (
            _mean_over_total(raw_tonal_stability, fill_value=0.0)
            if raw_tonal_stability is not None
            else None
        )
        contour_diversity_avg = (
            _mean_over_total(raw_contour_diversity, fill_value=0.0)
            if raw_contour_diversity is not None
            else None
        )
        ascending_tendency_avg = (
            _mean_over_total(raw_ascending_tendency, fill_value=0.0)
            if raw_ascending_tendency is not None
            else None
        )

        adherence_avg = (
            _mean_over_total(raw_adherence_text, fill_value=0.0)
            if raw_adherence_text is not None
            else None
        )
        adherence_is_fraction = None
        if raw_adherence_text is not None:
            ad_no_na = raw_adherence_text.dropna()
            if len(ad_no_na):
                adherence_is_fraction = bool(ad_no_na.ge(0).all() and ad_no_na.le(1).all())

        # Percentage of "scalar" style among compiled rows
        if raw_compiles is not None:
            compiled_mask = (raw_compiles == True)
            if compiled_mask.any():
                scalar_pct = float(
                    ((df_b.loc[compiled_mask, "style_label"] == "scalar").mean()) * 100.0
                )
            else:
                scalar_pct = None
        else:
            scalar_pct = None

        step_share_avg = _mean_dropna(df_b["step_share"]) if "step_share" in df_b.columns else None

        # note_count: mode over successful rows, ignoring 0 / NaN
        if "note_count" in df_b.columns:
            s_nc = pd.to_numeric(df_b["note_count"], errors="coerce")
            s_nc = s_nc[success_mask].dropna()
            s_nc = s_nc[s_nc > 0]
            if len(s_nc):
                nc_mode_series = s_nc.mode()
                note_count_mode = float(nc_mode_series.iloc[0]) if len(nc_mode_series) else None
            else:
                note_count_mode = None
        else:
            note_count_mode = None

        # bar_count_literal: mode over successful rows, ignoring 0 / NaN
        if "bar_count_literal" in df_b.columns:
            s_bc = pd.to_numeric(df_b["bar_count_literal"], errors="coerce")
            s_bc = s_bc[success_mask].dropna()
            s_bc = s_bc[s_bc > 0]
            if len(s_bc):
                bc_mode_series = s_bc.mode()
                bar_count_literal_mode = float(bc_mode_series.iloc[0]) if len(bc_mode_series) else None
            else:
                bar_count_literal_mode = None
        else:
            bar_count_literal_mode = None

        avg_row: Dict[str, Any] = {}
        for col in df_b.columns:
            if col == "batch":
                avg_row[col] = "AVG"
                continue
            if col == "status":
                avg_row[col] = f"{ok_pct:.1f}% ok" if ok_pct is not None else None
                continue
            if col == "run":
                avg_row[col] = n_runs
                continue

            if col == "compiles":
                avg_row[col] = f"{compiled_rate * 100:.1f}%" if compiled_rate is not None else None
                continue
            if col in ("in_key_ok", "beats_ok", "bars_ok_musical", "drift_detected"):
                mapping = {
                    "in_key_ok": in_key_rate,
                    "beats_ok": beats_ok_rate,
                    "bars_ok_musical": bars_ok_rate,
                    "drift_detected": drift_detected_rate,
                }
                rate = mapping[col]
                avg_row[col] = f"{rate * 100:.1f}%" if rate is not None else None
                continue

            if col == "in_key_pct_time_weighted":
                avg_row[col] = (
                    f"{round(in_key_pct_time_weighted_avg * 100, 1)}%"
                    if in_key_pct_time_weighted_avg is not None
                    else None
                )
                continue
            if col == "beats_per_bar_match_rate":
                avg_row[col] = (
                    round(beats_per_bar_match_rate_avg, 3)
                    if beats_per_bar_match_rate_avg is not None
                    else None
                )
                continue
            if col == "tonal_stability_score":
                avg_row[col] = (
                    round(tonal_stability_avg, 3) if tonal_stability_avg is not None else None
                )
                continue
            if col == "contour_diversity":
                avg_row[col] = (
                    round(contour_diversity_avg, 3) if contour_diversity_avg is not None else None
                )
                continue
            if col == "ascending_tendency":
                avg_row[col] = (
                    round(ascending_tendency_avg, 3)
                    if ascending_tendency_avg is not None
                    else None
                )
                continue
            if col == "adherence_text":
                if adherence_avg is None:
                    avg_row[col] = None
                else:
                    if adherence_is_fraction:
                        avg_row[col] = f"{round(adherence_avg * 100, 1)}%"
                    else:
                        avg_row[col] = round(adherence_avg, 3)
                continue

            if col == "style_label":
                avg_row[col] = f"{scalar_pct:.1f}% scalar" if scalar_pct is not None else None
                continue

            if col == "note_count":
                avg_row[col] = int(note_count_mode) if note_count_mode is not None else None
                continue
            if col == "bar_count_literal":
                avg_row[col] = int(bar_count_literal_mode) if bar_count_literal_mode is not None else None
                continue

            series_col = df_b[col]
            if col in num_dropna:
                avg_row[col] = _mean_dropna(series_col)
            elif col in pct_total:
                avg_row[col] = _mean_over_total(series_col, fill_value=0.0)
            elif col in bool_total:
                rate = _rate_over_total(series_col, missing_as=None)
                avg_row[col] = f"{rate * 100:.1f}%" if rate is not None else None
            else:
                avg_row[col] = _mean_dropna(series_col)

        if step_share_avg is not None:
            avg_row["step_share"] = round(step_share_avg, 3)

        df_disp_avg = pd.concat([df_disp, pd.DataFrame([avg_row])], ignore_index=True)

        # ---------- Pretty display ----------

        try:
            int_cols_present = [
                c for c in ["tokens", "note_count", "bar_count_literal"] if c in df_disp_avg.columns
            ]
            fmt_int = {c: _fmt_int_optional for c in int_cols_present}

            fmt3_cols = [
                c
                for c in [
                    "gen_s",
                    "interval_entropy",
                    "note_density",
                    "avg_interval_size",
                    "repeat_rate",
                    "beats_per_bar_match_rate",
                    "step_share",
                    "step_vs_leap",
                    "tonal_stability_score",
                    "contour_diversity",
                    "ascending_tendency",
                ]
                if c in df_disp_avg.columns
            ]
            fmt_generic = {c: _fmt3_optional for c in fmt3_cols}

            display(Markdown(f"### Batch: **{batch}**  (n={len(df_b)})"))
            display(df_disp_avg.style.format({**fmt_int, **fmt_generic}, na_rep=""))
        except Exception:
            display(Markdown(f"### Batch: **{batch}**  (n={len(df_b)})"))
            display(df_disp_avg)

        # ---------- Save AVG row to JSON ----------

        def _jsonify_row(row: Dict[str, Any]) -> Dict[str, Any]:
            # Convert a row dict into a JSON-serializable payload.
            out: Dict[str, Any] = {}
            for key, value in row.items():
                if value is None:
                    out[key] = None
                    continue
                try:
                    if pd.isna(value):
                        out[key] = None
                        continue
                except Exception:
                    pass
                try:
                    if hasattr(pd, "Int64Dtype") and isinstance(value, pd.Int64Dtype().type):
                        out[key] = int(value)
                        continue
                except Exception:
                    pass
                if isinstance(value, pd.Timestamp):
                    out[key] = value.isoformat()
                else:
                    out[key] = value
            return out

        avg_row_plain = _jsonify_row(avg_row)
        (final_root / batch).mkdir(parents=True, exist_ok=True)
        (final_root / batch / "AVG_row.json").write_text(
            json.dumps(avg_row_plain, indent=2, ensure_ascii=False),
            encoding="utf-8",
        )
        avg_rows_by_batch[batch] = avg_row_plain

        dfs_by_batch[batch] = df_disp_avg

    return dfs_by_batch, avg_rows_by_batch


**LilyPond compiler:** `C:\lilypond-2.24.4-mingw-x86_64\lilypond-2.24.4\bin\lilypond.exe`

--------------------------

<div style="background-color:#f2f2f2;padding:10px;border-radius:8px;">
  <h3 style="color:black;">Key</h3>
</div>

##  C Major

In [106]:
dfs = run_full_pipeline_for_all(only_batches=["c_major"])

### Batch: **c_major**  (n=20)

,batch,run,compiles,status,tokens,gen_s,bars_ok_musical,beats_ok,beats_per_bar_match_rate,interval_entropy,step_share,step_vs_leap,avg_interval_size,direction_changes,repeat_rate,note_density,in_key_ok,in_key_pct_time_weighted,tonal_stability_score,drift_detected,contour_diversity,ascending_tendency,style_label,adherence_text,note_count,bar_count_literal
0,c_major,1,✅ Yes,ok,1589,11.716,True,True,1.000,1.823,1.000,1.000,1.742,3.000000,0.000,4.000,True,100.0%,0.000,True,0.133,0.516,balanced,100.0%,32,8
1,c_major,2,✅ Yes,ok,2541,16.136,True,True,1.000,1.355,0.871,0.871,2.742,8.000000,0.000,4.000,True,100.0%,0.000,True,0.100,0.871,balanced,100.0%,32,8
2,c_major,3,✅ Yes,ok,1431,8.480,True,True,1.000,2.142,1.000,1.000,1.548,0.000000,0.097,4.000,True,100.0%,0.000,True,0.200,0.500,balanced,100.0%,32,8
3,c_major,4,✅ Yes,ok,1589,10.067,True,True,1.000,1.238,0.903,0.903,2.710,6.000000,0.000,4.000,True,100.0%,0.000,True,0.100,0.903,balanced,100.0%,32,8
4,c_major,5,✅ Yes,ok,1297,38.709,True,True,1.000,2.142,1.000,1.000,1.548,0.000000,0.097,4.000,True,100.0%,0.000,True,0.200,0.500,balanced,100.0%,32,8
5,c_major,6,✅ Yes,ok,2659,15.149,True,True,1.000,2.188,1.000,1.000,1.484,1.000000,0.129,4.000,True,100.0%,0.000,True,0.200,0.519,balanced,100.0%,32,8
6,c_major,7,✅ Yes,ok,2245,13.876,True,True,1.000,1.482,0.774,0.774,2.419,14.000000,0.000,4.000,True,100.0%,1.000,False,0.100,0.774,balanced,100.0%,32,8
7,c_major,8,✅ Yes,ok,2053,12.515,True,True,1.000,2.436,0.903,0.903,2.097,4.000000,0.032,4.000,True,100.0%,0.000,True,0.200,0.467,balanced,100.0%,32,8
8,c_major,9,✅ Yes,ok,1627,10.673,True,True,1.000,2.060,0.935,0.935,2.194,5.000000,0.032,4.000,True,100.0%,0.000,True,0.200,0.700,balanced,100.0%,32,8
9,c_major,10,✅ Yes,ok,1213,7.057,True,True,1.000,2.060,0.935,0.935,2.194,5.000000,0.032,4.000,True,100.0%,0.000,True,0.200,0.700,balanced,100.0%,32,8


### Observation
The C-major samples look solid overall. Everything compiles, the bars and beats line up cleanly, and the melodies stay fully in key. The model moves around more than in the simpler relative setups, with bigger intervals and more direction changes, so the lines feel a bit more active and less pattern-like. There’s still some wandering here and there, but nothing dramatic. In general, C-major gives stable, well-formed results with a bit more musical variety than the very plain baseline.

-----------------

##  F Major

In [107]:
dfs = run_full_pipeline_for_all(only_batches=["f_major"])

### Batch: **f_major**  (n=20)

,batch,run,compiles,status,tokens,gen_s,bars_ok_musical,beats_ok,beats_per_bar_match_rate,interval_entropy,step_share,step_vs_leap,avg_interval_size,direction_changes,repeat_rate,note_density,in_key_ok,in_key_pct_time_weighted,tonal_stability_score,drift_detected,contour_diversity,ascending_tendency,style_label,adherence_text,note_count,bar_count_literal
0,f_major,1,✅ Yes,ok,1773,18.018,True,True,1.000,1.355,0.871,0.871,2.742,8.000000,0.000,4.000,True,100.0%,0.000,True,0.100,0.871,balanced,100.0%,32,8
1,f_major,2,✅ Yes,ok,1832,10.176,True,True,1.000,2.412,0.839,0.839,2.645,8.000000,0.000,4.000,True,100.0%,1.000,False,0.133,0.419,balanced,100.0%,32,8
2,f_major,3,✅ Yes,ok,1748,8.951,True,True,1.000,2.207,0.903,0.903,2.387,6.000000,0.000,4.000,True,100.0%,0.000,True,0.133,0.645,balanced,100.0%,32,8
3,f_major,4,✅ Yes,ok,4095,27.033,True,True,1.000,1.157,0.871,0.871,3.000,8.000000,0.000,4.000,True,100.0%,0.000,True,0.100,0.871,balanced,100.0%,32,8
4,f_major,5,✅ Yes,ok,2577,15.188,True,True,1.000,1.355,0.871,0.871,2.742,8.000000,0.000,4.000,True,100.0%,0.000,True,0.100,0.871,balanced,100.0%,32,8
5,f_major,6,✅ Yes,ok,3830,20.918,True,True,1.000,1.642,0.710,0.710,6.226,8.000000,0.000,4.000,True,100.0%,0.000,True,0.100,0.871,balanced,100.0%,32,8
6,f_major,7,✅ Yes,ok,2840,12.161,True,True,1.000,1.423,0.871,0.871,2.968,8.000000,0.000,4.000,True,100.0%,0.750,False,0.100,0.871,balanced,100.0%,32,8
7,f_major,8,✅ Yes,ok,2042,10.897,True,True,1.000,1.157,0.871,0.871,3.000,8.000000,0.000,4.000,True,100.0%,0.000,True,0.100,0.871,balanced,100.0%,32,8
8,f_major,9,✅ Yes,ok,1261,5.912,True,True,1.000,1.355,0.871,0.871,2.742,8.000000,0.000,4.000,True,100.0%,0.000,True,0.100,0.871,balanced,100.0%,32,8
9,f_major,10,✅ Yes,ok,2662,16.328,True,True,1.000,1.447,0.871,0.871,2.581,8.000000,0.000,4.000,True,100.0%,0.750,True,0.100,0.871,balanced,100.0%,32,8


### Observation
The F-major runs are stable overall. Everything compiles, the rhythm is correct, and the lines stay fully in key. Compared to C-major, the melodies move around more, with bigger jumps and more frequent changes in direction, so the output feels a bit more energetic and less predictable. There’s still a bit of wandering, but nothing that breaks the structure. Overall, F-major works well and gives more movement while still keeping things clean.

-----------

##   E Minor

In [108]:
dfs = run_full_pipeline_for_all(only_batches=["e_minor"])

### Batch: **e_minor**  (n=20)

,batch,run,compiles,status,tokens,gen_s,bars_ok_musical,beats_ok,beats_per_bar_match_rate,interval_entropy,step_share,step_vs_leap,avg_interval_size,direction_changes,repeat_rate,note_density,in_key_ok,in_key_pct_time_weighted,tonal_stability_score,drift_detected,contour_diversity,ascending_tendency,style_label,adherence_text,note_count,bar_count_literal
0,e_minor,1,✅ Yes,ok,2580,15.940,True,True,1.000,1.157,0.871,0.871,3.000,8.000000,0.000,4.000,True,100.0%,0.500,True,0.100,0.871,balanced,100.0%,32,8
1,e_minor,2,❌ No,failed_to_compile,1870,11.492,,,,,,,,,,,,,,,,,,,,
2,e_minor,3,✅ Yes,ok,2936,16.131,True,True,1.000,1.355,0.871,0.871,2.742,8.000000,0.000,4.000,True,100.0%,0.500,True,0.100,0.871,balanced,100.0%,32,8
3,e_minor,4,✅ Yes,ok,2375,16.793,True,True,1.000,1.355,0.871,0.871,2.742,8.000000,0.000,4.000,True,100.0%,0.500,True,0.100,0.871,balanced,100.0%,32,8
4,e_minor,5,✅ Yes,ok,1924,11.325,True,True,1.000,1.482,0.774,0.774,2.419,14.000000,0.000,4.000,True,100.0%,0.000,True,0.100,0.774,balanced,100.0%,32,8
5,e_minor,6,✅ Yes,ok,2567,15.191,True,True,1.000,1.157,0.871,0.871,3.000,8.000000,0.000,4.000,True,100.0%,0.500,True,0.100,0.871,balanced,100.0%,32,8
6,e_minor,7,✅ Yes,ok,1550,9.059,True,True,1.000,2.524,0.806,0.806,3.032,11.000000,0.000,4.000,True,100.0%,0.500,True,0.133,0.484,balanced,100.0%,32,8
7,e_minor,8,✅ Yes,ok,2233,11.846,True,True,1.000,1.355,0.871,0.871,2.742,8.000000,0.000,4.000,True,100.0%,0.500,True,0.100,0.871,balanced,100.0%,32,8
8,e_minor,9,✅ Yes,ok,1745,9.710,True,True,1.000,2.607,0.871,0.871,2.581,8.000000,0.097,4.000,False,87.5%,0.000,True,0.267,0.500,balanced,100.0%,32,8
9,e_minor,10,✅ Yes,ok,2449,13.786,True,True,1.000,1.386,0.774,0.774,3.548,10.000000,0.000,4.000,True,100.0%,0.500,True,0.100,0.839,balanced,100.0%,32,8


### Observation
The E-minor samples are a bit less stable than the major keys. Most of them compile fine, but a few fail, and the bar and beat structure isn’t perfect every time. The melodies move around a lot, with plenty of direction changes and bigger intervals, and they don’t stay in key as consistently as C or F. Still, most outputs are usable, just a bit rougher and more restless compared to the major-key runs.


-----------

##   D Dorian

In [109]:
dfs = run_full_pipeline_for_all(only_batches=["d_dorian"])

### Batch: **d_dorian**  (n=20)

,batch,run,compiles,status,tokens,gen_s,bars_ok_musical,beats_ok,beats_per_bar_match_rate,interval_entropy,step_share,step_vs_leap,avg_interval_size,direction_changes,repeat_rate,note_density,in_key_ok,in_key_pct_time_weighted,tonal_stability_score,drift_detected,contour_diversity,ascending_tendency,style_label,adherence_text,note_count,bar_count_literal
0,d_dorian,1,✅ Yes,ok,1667,9.139,True,True,1.000,1.423,0.871,0.871,2.806,8.000000,0.000,4.000,True,100.0%,0.000,True,0.100,0.871,balanced,100.0%,32,8
1,d_dorian,2,✅ Yes,ok,2271,14.720,True,True,1.000,1.355,0.871,0.871,2.742,8.000000,0.000,4.000,True,100.0%,0.000,True,0.100,0.871,balanced,100.0%,32,8
2,d_dorian,3,✅ Yes,ok,2530,44.349,True,True,1.000,1.157,0.871,0.871,3.000,8.000000,0.000,4.000,True,100.0%,0.000,True,0.100,0.871,balanced,100.0%,32,8
3,d_dorian,4,✅ Yes,ok,2093,11.432,True,True,1.000,1.355,0.871,0.871,2.742,8.000000,0.000,4.000,True,100.0%,0.000,True,0.100,0.871,balanced,100.0%,32,8
4,d_dorian,5,✅ Yes,ok,1845,10.941,True,True,1.000,1.355,0.871,0.871,2.742,8.000000,0.000,4.000,True,100.0%,0.000,True,0.100,0.871,balanced,100.0%,32,8
5,d_dorian,6,✅ Yes,ok,3133,17.715,True,True,1.000,1.355,0.871,0.871,2.742,8.000000,0.000,4.000,True,100.0%,0.000,True,0.100,0.871,balanced,100.0%,32,8
6,d_dorian,7,✅ Yes,ok,2209,12.751,True,True,1.000,1.355,0.871,0.871,2.742,8.000000,0.000,4.000,True,100.0%,0.000,True,0.100,0.871,balanced,100.0%,32,8
7,d_dorian,8,❌ No,empty,4441,24.963,,,,,,,,,,,,,,,,,,,,
8,d_dorian,9,✅ Yes,ok,1996,9.930,True,True,1.000,2.317,0.871,0.871,2.774,7.000000,0.000,4.000,True,100.0%,0.000,True,0.133,0.484,balanced,100.0%,32,8
9,d_dorian,10,❌ No,empty,4441,23.675,,,,,,,,,,,,,,,,,,,,


### Observation
The D-dorian results look fairly close to the minor key behavior. Most samples compile, but not all, and the timing isn’t perfect every time. The melodies move quite a lot, with big intervals and plenty of direction changes, giving the lines a more active feel. It stays in the mode most of the time, though not as tightly as the major keys. Overall, it works reasonably well, just with a bit more movement and a few more small issues.

-----------

##  G Mixolydian

In [110]:
dfs = run_full_pipeline_for_all(only_batches=["g_mixolydian"])

### Batch: **g_mixolydian**  (n=20)

,batch,run,compiles,status,tokens,gen_s,bars_ok_musical,beats_ok,beats_per_bar_match_rate,interval_entropy,step_share,step_vs_leap,avg_interval_size,direction_changes,repeat_rate,note_density,in_key_ok,in_key_pct_time_weighted,tonal_stability_score,drift_detected,contour_diversity,ascending_tendency,style_label,adherence_text,note_count,bar_count_literal
0,g_mixolydian,1,✅ Yes,ok,1691,12.004,True,True,1.000,1.499,0.839,0.839,3.129,9.000000,0.000,4.000,True,100.0%,0.000,True,0.100,0.839,balanced,100.0%,32,8
1,g_mixolydian,2,✅ Yes,ok,3245,20.068,True,True,1.000,1.157,0.839,0.839,3.323,9.000000,0.000,4.000,True,100.0%,0.000,True,0.100,0.839,balanced,100.0%,32,8
2,g_mixolydian,3,❌ No,failed_to_compile,2473,13.785,,,,,,,,,,,,,,,,,,,,
3,g_mixolydian,4,❌ No,failed_to_compile,3923,25.163,,,,,,,,,,,,,,,,,,,,
4,g_mixolydian,5,✅ Yes,ok,2954,15.359,True,True,1.000,1.499,0.839,0.839,3.129,9.000000,0.000,4.000,True,100.0%,0.000,True,0.100,0.839,balanced,100.0%,32,8
5,g_mixolydian,6,✅ Yes,ok,2646,16.481,True,True,1.000,1.423,0.871,0.871,2.806,8.000000,0.000,4.000,True,100.0%,0.000,True,0.100,0.871,balanced,100.0%,32,8
6,g_mixolydian,7,✅ Yes,ok,1768,9.159,True,True,1.000,1.157,0.871,0.871,3.000,8.000000,0.000,4.000,True,100.0%,0.000,True,0.100,0.871,balanced,100.0%,32,8
7,g_mixolydian,8,✅ Yes,ok,1729,10.959,True,True,1.000,1.157,0.839,0.839,3.323,9.000000,0.000,4.000,True,100.0%,0.000,True,0.100,0.839,balanced,100.0%,32,8
8,g_mixolydian,9,✅ Yes,ok,1716,29.396,True,True,1.000,1.157,0.839,0.839,3.323,9.000000,0.000,4.000,True,100.0%,0.000,True,0.100,0.839,balanced,100.0%,32,8
9,g_mixolydian,10,✅ Yes,ok,1473,10.625,True,True,1.000,1.157,0.839,0.839,3.323,9.000000,0.000,4.000,True,100.0%,0.000,True,0.100,0.839,balanced,100.0%,32,8


### Observation
The G-mixolydian outputs are a bit less reliable than the other modes. Most of them compile, but not all, and the bar and beat structure isn’t always correct. The melodies jump around quite a lot, with bigger intervals and frequent direction changes, which gives them a more restless feel. They stay in the mode most of the time, but not perfectly. Overall, the results are usable, just a little rougher and less consistent than the major and minor keys.

-----------

##  G Major (1♯)

In [111]:
dfs = run_full_pipeline_for_all(only_batches=["g_major"])

### Batch: **g_major**  (n=20)

,batch,run,compiles,status,tokens,gen_s,bars_ok_musical,beats_ok,beats_per_bar_match_rate,interval_entropy,step_share,step_vs_leap,avg_interval_size,direction_changes,repeat_rate,note_density,in_key_ok,in_key_pct_time_weighted,tonal_stability_score,drift_detected,contour_diversity,ascending_tendency,style_label,adherence_text,note_count,bar_count_literal
0,g_major,1,✅ Yes,ok,1998,11.193,True,True,1.000,1.999,0.581,0.581,5.968,17.000000,0.000,4.000,False,87.5%,0.250,True,0.100,0.710,balanced,90.0%,32,8
1,g_major,2,✅ Yes,ok,1863,9.589,True,True,1.000,1.355,0.871,0.871,2.742,8.000000,0.000,4.000,True,100.0%,0.000,True,0.100,0.871,balanced,100.0%,32,8
2,g_major,3,❌ No,failed_to_compile,1480,6.751,,,,,,,,,,,,,,,,,,,,
3,g_major,4,✅ Yes,ok,3114,18.215,True,True,1.000,2.848,0.839,0.839,2.355,5.000000,0.129,4.000,True,100.0%,0.500,True,0.267,0.481,balanced,100.0%,32,8
4,g_major,5,✅ Yes,ok,2728,16.247,True,True,1.000,1.501,0.613,0.613,5.839,16.000000,0.000,4.000,False,87.5%,0.250,True,0.100,0.742,balanced,90.0%,32,8
5,g_major,6,✅ Yes,ok,1493,8.184,True,True,1.000,1.157,0.871,0.871,3.000,8.000000,0.000,4.000,True,100.0%,0.000,True,0.100,0.871,balanced,100.0%,32,8
6,g_major,7,✅ Yes,ok,2151,11.025,True,True,1.000,2.193,0.839,0.839,2.452,8.000000,0.000,4.000,True,100.0%,0.250,True,0.133,0.710,balanced,100.0%,32,8
7,g_major,8,✅ Yes,ok,3535,30.299,True,True,1.000,1.355,0.871,0.871,2.742,8.000000,0.000,4.000,True,100.0%,0.000,True,0.100,0.871,balanced,100.0%,32,8
8,g_major,9,✅ Yes,ok,3114,18.327,True,True,1.000,1.499,0.839,0.839,3.129,9.000000,0.000,4.000,True,100.0%,0.000,True,0.100,0.839,balanced,100.0%,32,8
9,g_major,10,✅ Yes,ok,1427,7.412,True,True,1.000,1.355,0.581,0.581,6.161,17.000000,0.000,4.000,False,87.5%,0.250,True,0.100,0.710,balanced,90.0%,32,8


### Observation
The G-major runs are mostly stable. Most samples compile, the timing is usually correct, and the lines stay in key most of the time. The melodies move quite a lot, with bigger jumps and regular direction changes, so they feel active and a bit less predictable. Overall, the results are good, just not as tight as the cleaner keys.

----------------

##  D Major (2♯)

In [112]:
dfs = run_full_pipeline_for_all(only_batches=["d_major"])

### Batch: **d_major**  (n=20)

,batch,run,compiles,status,tokens,gen_s,bars_ok_musical,beats_ok,beats_per_bar_match_rate,interval_entropy,step_share,step_vs_leap,avg_interval_size,direction_changes,repeat_rate,note_density,in_key_ok,in_key_pct_time_weighted,tonal_stability_score,drift_detected,contour_diversity,ascending_tendency,style_label,adherence_text,note_count,bar_count_literal
0,d_major,1,✅ Yes,ok,1667,12.931,True,True,1.000,2.415,0.903,0.903,2.387,6.000000,0.065,4.000,False,81.2%,0.000,True,0.200,0.483,balanced,100.0%,32,8
1,d_major,2,✅ Yes,ok,2072,11.663,True,True,1.000,1.157,0.871,0.871,3.000,8.000000,0.000,4.000,True,100.0%,0.000,True,0.100,0.871,balanced,100.0%,32,8
2,d_major,3,✅ Yes,ok,2410,14.600,True,True,1.000,1.355,0.871,0.871,2.742,8.000000,0.000,4.000,True,100.0%,0.000,True,0.100,0.871,balanced,100.0%,32,8
3,d_major,4,✅ Yes,ok,2499,13.911,True,True,1.000,1.355,0.871,0.871,2.742,8.000000,0.000,4.000,True,100.0%,0.000,True,0.100,0.871,balanced,100.0%,32,8
4,d_major,5,✅ Yes,ok,1931,11.491,True,True,1.000,2.020,0.645,0.645,5.097,15.000000,0.000,4.000,False,93.8%,0.000,True,0.133,0.710,balanced,100.0%,32,8
5,d_major,6,✅ Yes,ok,3046,17.971,True,True,1.000,1.230,0.839,0.839,2.968,10.000000,0.000,4.000,True,100.0%,0.500,True,0.100,0.839,balanced,100.0%,32,8
6,d_major,7,✅ Yes,ok,1934,11.620,True,True,1.000,1.355,0.871,0.871,2.742,8.000000,0.000,4.000,True,100.0%,0.000,True,0.100,0.871,balanced,100.0%,32,8
7,d_major,8,✅ Yes,ok,2464,17.236,True,True,1.000,1.850,0.419,0.419,9.645,18.000000,0.000,4.000,False,71.9%,0.000,True,0.100,0.710,balanced,90.0%,32,8
8,d_major,9,✅ Yes,ok,2216,13.450,True,True,1.000,1.355,0.871,0.871,2.742,8.000000,0.000,4.000,True,100.0%,0.000,True,0.100,0.871,balanced,100.0%,32,8
9,d_major,10,✅ Yes,ok,2282,12.111,True,True,1.000,1.230,0.839,0.839,2.968,10.000000,0.000,4.000,True,100.0%,0.500,True,0.100,0.839,balanced,100.0%,32,8


### Observation
The D-major samples behave well. Everything compiles, the rhythm lines up cleanly, and the lines stay mostly in key. The melodies have plenty of movement, with both steps and bigger intervals, but they stay controlled. Overall, D-major gives solid and consistent outputs.

----------------

##  B♭ Major (2♭)

In [7]:
dfs = run_full_pipeline_for_all(only_batches=["bes_major"])

### Batch: **bes_major**  (n=20)

,batch,run,compiles,status,tokens,gen_s,bars_ok_musical,beats_ok,beats_per_bar_match_rate,interval_entropy,step_share,step_vs_leap,avg_interval_size,direction_changes,repeat_rate,note_density,in_key_ok,in_key_pct_time_weighted,tonal_stability_score,drift_detected,contour_diversity,ascending_tendency,style_label,adherence_text,note_count,bar_count_literal
0,bes_major,1,✅ Yes,lily_only_ok,2477,18.549,,,,,,,,,,,,,,,,,,90.0%,,
1,bes_major,2,✅ Yes,ok,1947,12.466,True,True,1.000,1.355,0.871,0.871,2.742,8.000000,0.000,4.000,True,100.0%,0.250,True,0.100,0.871,balanced,90.0%,32,8
2,bes_major,3,✅ Yes,ok,2222,12.064,True,True,1.000,1.767,0.774,0.774,2.710,6.000000,0.000,4.000,False,87.5%,1.000,False,0.100,0.903,balanced,90.0%,32,8
3,bes_major,4,✅ Yes,ok,3822,26.645,True,True,1.000,2.624,0.774,0.774,3.065,7.000000,0.000,4.000,True,100.0%,0.250,True,0.133,0.645,balanced,90.0%,32,8
4,bes_major,5,✅ Yes,ok,1144,6.372,True,True,1.000,1.418,0.839,0.839,3.000,9.000000,0.000,4.000,True,100.0%,0.000,True,0.100,0.839,balanced,90.0%,32,8
5,bes_major,6,✅ Yes,ok,1867,10.708,True,True,1.000,2.231,0.935,0.935,2.194,3.000000,0.065,4.000,True,100.0%,0.000,True,0.267,0.310,balanced,90.0%,32,8
6,bes_major,7,✅ Yes,ok,2290,14.762,True,True,1.000,2.139,0.935,0.935,2.258,5.000000,0.000,4.000,False,87.5%,0.000,True,0.133,0.516,balanced,100.0%,32,8
7,bes_major,8,✅ Yes,ok,1887,11.783,True,True,1.000,2.142,1.000,1.000,1.548,0.000000,0.097,4.000,True,100.0%,0.000,True,0.200,0.500,balanced,90.0%,32,8
8,bes_major,9,✅ Yes,ok,2522,12.959,True,True,1.000,2.176,0.871,0.871,2.742,8.000000,0.000,4.000,True,100.0%,0.000,True,0.133,0.323,balanced,90.0%,32,8
9,bes_major,10,✅ Yes,ok,2374,18.330,True,True,1.000,1.947,0.968,0.968,1.968,5.000000,0.000,4.000,False,84.4%,0.250,True,0.133,0.645,balanced,90.0%,32,8


### Observation
The B-flat major results are mostly stable. Almost all samples compile and the rhythm is usually correct, though a few runs slip. The melodies move around a fair bit, with mixed intervals and some direction changes, and they don’t stay in key as tightly as the sharp keys. Still, most of the structure holds up, and the outputs feel reasonably consistent, just a little softer and less precise compared to the cleaner major-key cases.

----------------

##  A Minor (0)

In [114]:
dfs = run_full_pipeline_for_all(only_batches=["a_minor"])

### Batch: **a_minor**  (n=20)

,batch,run,compiles,status,tokens,gen_s,bars_ok_musical,beats_ok,beats_per_bar_match_rate,interval_entropy,step_share,step_vs_leap,avg_interval_size,direction_changes,repeat_rate,note_density,in_key_ok,in_key_pct_time_weighted,tonal_stability_score,drift_detected,contour_diversity,ascending_tendency,style_label,adherence_text,note_count,bar_count_literal
0,a_minor,1,✅ Yes,ok,1955,9.743,True,True,1.000,1.906,0.871,0.871,3.032,9.000000,0.000,4.000,True,100.0%,0.500,True,0.133,0.323,balanced,100.0%,32,8
1,a_minor,2,✅ Yes,ok,2068,10.813,True,True,1.000,2.219,0.871,0.871,2.968,8.000000,0.000,4.000,True,100.0%,0.500,True,0.133,0.516,balanced,100.0%,32,8
2,a_minor,3,✅ Yes,ok,3386,19.442,True,True,1.000,1.157,0.839,0.839,3.323,10.000000,0.000,4.000,True,100.0%,0.500,True,0.100,0.839,balanced,100.0%,32,8
3,a_minor,4,✅ Yes,ok,1993,11.777,True,True,1.000,1.967,0.871,0.871,2.968,9.000000,0.000,4.000,True,100.0%,0.500,True,0.133,0.323,balanced,100.0%,32,8
4,a_minor,5,✅ Yes,ok,1947,11.787,True,True,1.000,1.418,0.839,0.839,3.000,9.000000,0.000,4.000,True,100.0%,0.500,True,0.100,0.839,balanced,100.0%,32,8
5,a_minor,6,✅ Yes,ok,2603,20.718,True,True,1.000,2.541,0.710,0.710,3.355,13.000000,0.000,4.000,True,100.0%,0.500,True,0.133,0.387,balanced,100.0%,32,8
6,a_minor,7,✅ Yes,ok,1659,10.231,True,True,1.000,1.418,0.839,0.839,3.000,9.000000,0.000,4.000,True,100.0%,0.500,True,0.100,0.839,balanced,100.0%,32,8
7,a_minor,8,✅ Yes,ok,2708,40.243,True,True,1.000,2.081,0.871,0.871,3.032,10.000000,0.000,4.000,True,100.0%,0.500,True,0.133,0.516,balanced,100.0%,32,8
8,a_minor,9,✅ Yes,ok,1448,8.311,True,True,1.000,2.552,0.742,0.742,3.419,12.000000,0.000,4.000,True,100.0%,0.500,True,0.133,0.645,balanced,100.0%,32,8
9,a_minor,10,✅ Yes,ok,1536,9.176,True,True,1.000,2.739,0.710,0.710,3.484,10.000000,0.000,4.000,True,100.0%,0.250,True,0.133,0.581,balanced,100.0%,32,8


### Observation
The A-minor samples behave well overall. Everything compiles cleanly, the rhythm is correct, and the lines stay fully in key. The melodies move quite a lot, with bigger intervals and plenty of direction changes, so they feel a bit more active than the major-key cases. Even with all that movement, the structure holds together without issues. Overall, A minor gives stable and lively results.

----------------

##  D Minor (1♭)

In [115]:
dfs = run_full_pipeline_for_all(only_batches=["d_minor"])

### Batch: **d_minor**  (n=20)

,batch,run,compiles,status,tokens,gen_s,bars_ok_musical,beats_ok,beats_per_bar_match_rate,interval_entropy,step_share,step_vs_leap,avg_interval_size,direction_changes,repeat_rate,note_density,in_key_ok,in_key_pct_time_weighted,tonal_stability_score,drift_detected,contour_diversity,ascending_tendency,style_label,adherence_text,note_count,bar_count_literal
0,d_minor,1,✅ Yes,ok,2261,15.730,True,True,1.000,1.355,0.871,0.871,2.742,8.000000,0.000,4.000,True,100.0%,0.500,True,0.100,0.871,balanced,100.0%,32,8
1,d_minor,2,✅ Yes,ok,2352,11.437,True,True,1.000,1.482,0.774,0.774,2.419,14.000000,0.000,4.000,True,100.0%,0.000,True,0.100,0.774,balanced,100.0%,32,8
2,d_minor,3,✅ Yes,ok,2561,17.267,True,True,1.000,1.680,0.871,0.871,2.710,10.000000,0.032,4.000,True,100.0%,0.750,True,0.167,0.833,balanced,100.0%,32,8
3,d_minor,4,✅ Yes,ok,2154,15.899,True,True,1.000,1.355,0.871,0.871,2.742,8.000000,0.000,4.000,True,100.0%,0.500,True,0.100,0.871,balanced,100.0%,32,8
4,d_minor,5,✅ Yes,ok,1665,12.165,True,True,1.000,2.317,0.871,0.871,2.774,7.000000,0.000,4.000,True,100.0%,0.500,True,0.133,0.484,balanced,100.0%,32,8
5,d_minor,6,✅ Yes,ok,2699,18.717,True,True,1.000,1.355,0.871,0.871,2.742,8.000000,0.000,4.000,True,100.0%,0.500,True,0.100,0.871,balanced,100.0%,32,8
6,d_minor,7,✅ Yes,ok,2793,22.071,True,True,1.000,2.474,0.871,0.871,2.613,9.000000,0.032,4.000,True,100.0%,0.750,False,0.200,0.400,balanced,100.0%,32,8
7,d_minor,8,✅ Yes,ok,2670,17.742,True,True,1.000,1.355,0.871,0.871,2.742,8.000000,0.000,4.000,True,100.0%,0.500,True,0.100,0.871,balanced,100.0%,32,8
8,d_minor,9,✅ Yes,ok,2239,14.350,True,True,1.000,1.355,0.871,0.871,2.742,8.000000,0.000,4.000,True,100.0%,0.500,True,0.100,0.871,balanced,100.0%,32,8
9,d_minor,10,✅ Yes,ok,2283,13.877,True,True,1.000,1.355,0.871,0.871,2.742,8.000000,0.000,4.000,True,100.0%,0.500,True,0.100,0.871,balanced,100.0%,32,8


### Observation
The D-minor samples look good overall. Almost all of them compile, the rhythm lines up well, and the music stays in key most of the time. The melodies move around a fair bit, with bigger jumps and regular changes in direction, so they feel pretty active. In general, the results are consistent and clean, just a little less tight than the best major-key runs.

----------------

##  C Lydian (modal, 1♯)

In [116]:
dfs = run_full_pipeline_for_all(only_batches=["c_lydian"])

### Batch: **c_lydian**  (n=20)

,batch,run,compiles,status,tokens,gen_s,bars_ok_musical,beats_ok,beats_per_bar_match_rate,interval_entropy,step_share,step_vs_leap,avg_interval_size,direction_changes,repeat_rate,note_density,in_key_ok,in_key_pct_time_weighted,tonal_stability_score,drift_detected,contour_diversity,ascending_tendency,style_label,adherence_text,note_count,bar_count_literal
0,c_lydian,1,✅ Yes,ok,2114,13.678,True,True,1.000,2.142,1.000,1.000,1.548,0.000000,0.097,4.000,True,100.0%,0.000,True,0.200,0.500,balanced,100.0%,32,8
1,c_lydian,2,✅ Yes,ok,1912,13.637,True,True,1.000,2.060,0.935,0.935,2.194,5.000000,0.032,4.000,True,100.0%,0.000,True,0.200,0.700,balanced,100.0%,32,8
2,c_lydian,3,✅ Yes,ok,3004,18.752,True,True,1.000,1.238,0.903,0.903,2.710,6.000000,0.000,4.000,True,100.0%,0.000,True,0.100,0.903,balanced,100.0%,32,8
3,c_lydian,4,✅ Yes,ok,2411,13.567,True,True,1.000,2.008,0.935,0.935,2.323,7.000000,0.000,4.000,True,100.0%,0.000,True,0.133,0.387,balanced,100.0%,32,8
4,c_lydian,5,✅ Yes,ok,2767,16.192,True,True,1.000,0.771,0.774,0.774,2.903,14.000000,0.000,4.000,True,100.0%,0.000,True,0.100,0.774,balanced,100.0%,32,8
5,c_lydian,6,✅ Yes,ok,2405,14.486,True,True,1.000,2.012,0.903,0.903,2.613,7.000000,0.000,4.000,True,100.0%,0.000,True,0.133,0.323,balanced,100.0%,32,8
6,c_lydian,7,✅ Yes,ok,1922,11.409,True,True,1.000,2.071,0.935,0.935,1.871,6.000000,0.000,4.000,True,100.0%,0.000,True,0.133,0.548,balanced,100.0%,32,8
7,c_lydian,8,✅ Yes,ok,1513,8.468,True,True,1.000,2.105,0.903,0.903,2.548,8.000000,0.000,4.000,True,100.0%,0.000,True,0.133,0.677,balanced,100.0%,32,8
8,c_lydian,9,✅ Yes,ok,1608,9.427,True,True,1.000,1.238,0.903,0.903,2.710,6.000000,0.000,4.000,True,100.0%,0.000,True,0.100,0.903,balanced,100.0%,32,8
9,c_lydian,10,❌ No,failed_to_compile,2952,16.783,,,,,,,,,,,,,,,,,,,,


### Observation
The C-lydian runs work pretty well overall. Most samples compile, the timing is mostly right, and the lines stay in the mode for the most part. The melodies move a fair bit, with both small steps and some bigger jumps, but they stay fairly controlled. In general, the outputs are steady and usable, just a little less clean than the best-performing keys.

----------------

<div style="background-color:#f2f2f2;padding:10px;border-radius:8px;">
  <h3 style="color:black;">Tempo</h3>
</div>

##   Slow Tempo

In [117]:
dfs = run_full_pipeline_for_all(only_batches=["slow_tempo"])

### Batch: **slow_tempo**  (n=20)

,batch,run,compiles,status,tokens,gen_s,bars_ok_musical,beats_ok,beats_per_bar_match_rate,interval_entropy,step_share,step_vs_leap,avg_interval_size,direction_changes,repeat_rate,note_density,in_key_ok,in_key_pct_time_weighted,tonal_stability_score,drift_detected,contour_diversity,ascending_tendency,style_label,adherence_text,note_count,bar_count_literal
0,slow_tempo,1,✅ Yes,ok,1918,12.459,True,True,1.000,2.142,1.000,1.000,1.548,0.000000,0.097,4.000,True,100.0%,0.000,True,0.200,0.500,balanced,100.0%,32,8
1,slow_tempo,2,✅ Yes,ok,1873,10.136,True,True,1.000,2.230,0.903,0.903,2.613,6.000000,0.000,4.000,True,100.0%,0.000,True,0.133,0.516,balanced,100.0%,32,8
2,slow_tempo,3,✅ Yes,ok,1878,9.503,True,True,1.000,1.238,0.903,0.903,2.710,6.000000,0.000,4.000,True,100.0%,0.000,True,0.100,0.903,balanced,100.0%,32,8
3,slow_tempo,4,✅ Yes,ok,1763,9.930,True,True,1.000,1.238,0.903,0.903,2.710,6.000000,0.000,4.000,True,100.0%,0.000,True,0.100,0.903,balanced,100.0%,32,8
4,slow_tempo,5,✅ Yes,ok,2723,18.903,True,True,1.000,2.142,1.000,1.000,1.548,0.000000,0.097,4.000,True,100.0%,0.000,True,0.200,0.500,balanced,100.0%,32,8
5,slow_tempo,6,✅ Yes,ok,1051,6.715,True,True,1.000,2.142,1.000,1.000,1.548,0.000000,0.097,4.000,True,100.0%,0.000,True,0.200,0.500,balanced,100.0%,32,8
6,slow_tempo,7,✅ Yes,ok,2435,15.332,True,True,1.000,3.256,0.613,0.613,2.258,1.000000,0.129,4.000,True,100.0%,0.250,True,0.233,0.519,balanced,100.0%,32,8
7,slow_tempo,8,✅ Yes,ok,1160,6.071,True,True,1.000,1.238,0.903,0.903,2.710,6.000000,0.000,4.000,True,100.0%,0.000,True,0.100,0.903,balanced,100.0%,32,8
8,slow_tempo,9,✅ Yes,ok,1203,5.881,True,True,1.000,1.482,0.774,0.774,2.419,14.000000,0.000,4.000,True,100.0%,1.000,False,0.100,0.774,balanced,100.0%,32,8
9,slow_tempo,10,✅ Yes,ok,1268,6.869,True,True,1.000,2.142,1.000,1.000,1.548,0.000000,0.097,4.000,True,100.0%,0.000,True,0.200,0.500,balanced,100.0%,32,8


### Observation
The slow-tempo samples come out clean and stable. Everything compiles, the timing is correct, and the melodies stay fully in key. The lines move a fair amount, with a mix of small steps and some bigger jumps, but the overall shape still feels controlled. Nothing major breaks, and the outputs look consistent across the batch.

-----------

##   Fast Tempo

In [118]:
dfs = run_full_pipeline_for_all(only_batches=["fast_tempo"])

### Batch: **fast_tempo**  (n=20)

,batch,run,compiles,status,tokens,gen_s,bars_ok_musical,beats_ok,beats_per_bar_match_rate,interval_entropy,step_share,step_vs_leap,avg_interval_size,direction_changes,repeat_rate,note_density,in_key_ok,in_key_pct_time_weighted,tonal_stability_score,drift_detected,contour_diversity,ascending_tendency,style_label,adherence_text,note_count,bar_count_literal
0,fast_tempo,1,✅ Yes,ok,1740,9.069,True,True,1.000,1.482,0.774,0.774,2.419,14.000000,0.000,4.000,True,100.0%,1.000,False,0.100,0.774,balanced,100.0%,32,8
1,fast_tempo,2,✅ Yes,ok,1669,9.230,True,True,1.000,2.317,0.935,0.935,1.935,2.000000,0.065,4.000,True,100.0%,0.000,True,0.167,0.517,balanced,100.0%,32,8
2,fast_tempo,3,✅ Yes,ok,1519,9.291,True,True,1.000,2.142,1.000,1.000,1.548,0.000000,0.097,4.000,True,100.0%,0.000,True,0.200,0.500,balanced,100.0%,32,8
3,fast_tempo,4,✅ Yes,ok,1283,7.534,True,True,1.000,2.230,0.968,0.968,1.774,3.000000,0.065,4.000,True,100.0%,0.250,True,0.200,0.517,balanced,100.0%,32,8
4,fast_tempo,5,✅ Yes,ok,2060,10.987,True,True,1.000,2.142,1.000,1.000,1.548,0.000000,0.097,4.000,True,100.0%,0.000,True,0.200,0.500,balanced,100.0%,32,8
5,fast_tempo,6,✅ Yes,ok,1682,8.670,True,True,1.000,2.142,1.000,1.000,1.548,0.000000,0.097,4.000,True,100.0%,0.000,True,0.200,0.500,balanced,100.0%,32,8
6,fast_tempo,7,✅ Yes,ok,2129,10.707,True,True,1.000,1.537,0.903,0.903,2.581,3.000000,0.065,4.000,True,100.0%,0.000,True,0.167,0.897,balanced,100.0%,32,8
7,fast_tempo,8,✅ Yes,ok,1588,8.740,True,True,1.000,2.060,0.935,0.935,2.194,5.000000,0.032,4.000,True,100.0%,0.000,True,0.200,0.700,balanced,100.0%,32,8
8,fast_tempo,9,✅ Yes,ok,1536,8.678,True,True,1.000,2.060,0.935,0.935,2.194,5.000000,0.032,4.000,True,100.0%,0.000,True,0.200,0.700,balanced,100.0%,32,8
9,fast_tempo,10,✅ Yes,ok,2580,15.680,True,True,1.000,2.060,0.935,0.935,2.194,5.000000,0.032,4.000,True,100.0%,0.000,True,0.200,0.700,balanced,100.0%,32,8


### Observation
The fast-tempo runs are stable and clean. Everything compiles, the rhythm lines up perfectly, and the melodies stay fully in key. The lines move a bit more quickly, with a good mix of steps and a few larger jumps, but they still feel controlled. Overall, the outputs look consistent and don’t show any major issues despite the higher speed.

-----------

<div style="background-color:#f2f2f2;padding:10px;border-radius:8px;">
  <h3 style="color:black;">Time Signature</h3>
</div>

##  3/4 (simple triple)

In [119]:
dfs = run_full_pipeline_for_all(only_batches=["three_four"])

### Batch: **three_four**  (n=20)

,batch,run,compiles,status,tokens,gen_s,bars_ok_musical,beats_ok,beats_per_bar_match_rate,interval_entropy,step_share,step_vs_leap,avg_interval_size,direction_changes,repeat_rate,note_density,in_key_ok,in_key_pct_time_weighted,tonal_stability_score,drift_detected,contour_diversity,ascending_tendency,style_label,adherence_text,note_count,bar_count_literal
0,three_four,1,✅ Yes,ok,2003,11.675,True,True,1.000,1.325,0.870,0.870,2.783,6.000000,0.000,3.000,True,100.0%,0.250,True,0.136,0.870,balanced,100.0%,24,8
1,three_four,2,✅ Yes,ok,1252,6.564,True,True,1.000,1.445,0.870,0.870,2.870,5.000000,0.000,3.000,True,100.0%,0.250,True,0.136,0.870,balanced,100.0%,24,8
2,three_four,3,✅ Yes,ok,1684,9.892,True,True,1.000,1.325,0.870,0.870,2.783,6.000000,0.000,3.000,True,100.0%,0.250,True,0.136,0.870,balanced,100.0%,24,8
3,three_four,4,✅ Yes,ok,1984,15.372,True,True,1.000,1.325,0.870,0.870,2.783,6.000000,0.000,3.000,True,100.0%,0.250,True,0.136,0.870,balanced,100.0%,24,8
4,three_four,5,✅ Yes,ok,2851,17.881,True,True,1.000,1.325,0.870,0.870,2.783,5.000000,0.000,3.000,True,100.0%,0.250,True,0.136,0.870,balanced,100.0%,24,8
5,three_four,6,✅ Yes,ok,2375,16.822,True,True,1.000,2.595,0.870,0.870,2.348,3.000000,0.130,3.000,True,100.0%,0.250,True,0.318,0.450,balanced,100.0%,24,8
6,three_four,7,✅ Yes,ok,1673,11.554,True,True,1.000,0.887,0.696,0.696,2.609,14.000000,0.000,3.000,True,100.0%,0.000,True,0.136,0.696,balanced,100.0%,24,8
7,three_four,8,✅ Yes,ok,2333,13.526,True,True,1.000,1.325,0.870,0.870,2.783,6.000000,0.000,3.000,True,100.0%,0.250,True,0.136,0.870,balanced,100.0%,24,8
8,three_four,9,✅ Yes,ok,1670,9.216,True,True,1.000,1.445,0.870,0.870,2.870,5.000000,0.000,3.000,True,100.0%,0.250,True,0.136,0.870,balanced,100.0%,24,8
9,three_four,10,✅ Yes,ok,1300,7.282,True,True,1.000,1.325,0.870,0.870,2.783,6.000000,0.000,3.000,True,100.0%,0.250,True,0.136,0.870,balanced,100.0%,24,8


### Observation
The 3/4 samples look clean and consistent. Everything compiles, the bar structure is correct, and the melodies stay fully in key. The lines have a bit of movement, with a mix of steps and some larger intervals, but nothing that breaks the flow. Overall, the model handles the triple meter without any trouble, and the outputs are steady across the batch.

##  6/8 (compound duple)

In [120]:
dfs = run_full_pipeline_for_all(only_batches=["six_eight"])

### Batch: **six_eight**  (n=20)

,batch,run,compiles,status,tokens,gen_s,bars_ok_musical,beats_ok,beats_per_bar_match_rate,interval_entropy,step_share,step_vs_leap,avg_interval_size,direction_changes,repeat_rate,note_density,in_key_ok,in_key_pct_time_weighted,tonal_stability_score,drift_detected,contour_diversity,ascending_tendency,style_label,adherence_text,note_count,bar_count_literal
0,six_eight,1,✅ Yes,ok,2244,13.957,True,True,1.000,1.337,0.872,0.872,2.745,12.000000,0.000,6.000,True,100.0%,0.500,True,0.065,0.872,balanced,100.0%,48,8
1,six_eight,2,✅ Yes,ok,2695,16.530,True,True,1.000,1.222,0.851,0.851,2.872,14.000000,0.000,6.000,True,100.0%,0.000,True,0.065,0.851,balanced,100.0%,48,8
2,six_eight,3,✅ Yes,ok,2243,13.329,True,True,1.000,1.337,0.872,0.872,2.745,12.000000,0.000,6.000,True,100.0%,0.500,True,0.065,0.872,balanced,100.0%,48,8
3,six_eight,4,✅ Yes,ok,2093,12.479,True,True,1.000,1.337,0.872,0.872,2.745,12.000000,0.000,6.000,True,100.0%,0.500,True,0.065,0.872,balanced,100.0%,48,8
4,six_eight,5,✅ Yes,ok,2376,14.203,True,True,1.000,1.222,0.851,0.851,2.872,14.000000,0.000,6.000,True,100.0%,0.000,True,0.065,0.851,balanced,100.0%,48,8
5,six_eight,6,✅ Yes,ok,2525,11.734,True,True,1.000,1.337,0.872,0.872,2.745,12.000000,0.000,6.000,True,100.0%,0.500,True,0.065,0.872,balanced,100.0%,48,8
6,six_eight,7,✅ Yes,ok,1288,6.038,True,True,1.000,1.222,0.851,0.851,2.872,14.000000,0.000,6.000,True,100.0%,0.000,True,0.065,0.851,balanced,100.0%,48,8
7,six_eight,8,✅ Yes,ok,2245,14.768,True,True,1.000,2.870,0.830,0.830,2.362,10.000000,0.149,6.000,True,100.0%,0.000,True,0.174,0.475,balanced,100.0%,48,8
8,six_eight,9,✅ Yes,ok,3052,31.495,True,True,1.000,1.337,0.872,0.872,2.745,12.000000,0.000,6.000,True,100.0%,0.500,True,0.065,0.872,balanced,100.0%,48,8
9,six_eight,10,✅ Yes,ok,1607,8.938,True,True,1.000,1.337,0.872,0.872,2.745,12.000000,0.000,6.000,True,100.0%,0.500,True,0.065,0.872,balanced,100.0%,48,8


### Observation
The 6/8 runs are stable and well-formed. Everything compiles, the bar and beat structure is correct, and the melodies stay fully in key. The lines move quite a lot, with frequent direction changes, but the overall flow still feels coherent. The model handles the compound meter smoothly, and the outputs are consistent across the batch.

-----------

<div style="background-color:#f2f2f2;padding:10px;border-radius:8px;">
  <h3 style="color:black;">Harmony </h3>
</div>

##  Dyads (2-note chords)

In [121]:
dfs = run_full_pipeline_for_all(only_batches=["dyads"])

### Batch: **dyads**  (n=20)

,batch,run,compiles,status,tokens,gen_s,bars_ok_musical,beats_ok,beats_per_bar_match_rate,interval_entropy,step_share,step_vs_leap,avg_interval_size,direction_changes,repeat_rate,note_density,in_key_ok,in_key_pct_time_weighted,tonal_stability_score,drift_detected,contour_diversity,ascending_tendency,style_label,adherence_text,note_count,bar_count_literal
0,dyads,1,✅ Yes,ok,3446,22.233,True,True,1.000,2.381,0.871,0.871,2.581,5.000000,0.000,8.000,True,100.0%,0.000,True,0.133,0.419,balanced,80.0%,64,
1,dyads,2,✅ Yes,ok,2408,14.357,True,True,1.000,1.482,0.774,0.774,2.419,14.000000,0.000,8.000,True,100.0%,0.000,True,0.100,0.774,balanced,80.0%,64,
2,dyads,3,✅ Yes,ok,3251,19.784,True,True,1.000,1.544,0.903,0.903,2.290,12.000000,0.000,8.000,True,100.0%,0.250,True,0.100,0.806,balanced,80.0%,64,
3,dyads,4,✅ Yes,ok,4498,29.588,True,True,1.000,1.642,0.871,0.871,2.484,16.000000,0.000,8.000,True,100.0%,0.250,True,0.100,0.742,balanced,80.0%,64,
4,dyads,5,✅ Yes,ok,4060,25.238,True,True,1.000,1.642,0.871,0.871,2.484,16.000000,0.000,8.000,True,100.0%,0.250,True,0.100,0.742,balanced,80.0%,64,
5,dyads,6,✅ Yes,ok,2918,17.529,True,True,1.000,1.642,0.871,0.871,2.484,8.000000,0.000,8.000,True,100.0%,0.500,True,0.133,0.742,balanced,80.0%,64,
6,dyads,7,✅ Yes,ok,3443,24.128,True,True,1.000,1.642,0.871,0.871,2.484,16.000000,0.000,8.000,True,100.0%,0.250,True,0.100,0.742,balanced,80.0%,64,
7,dyads,8,✅ Yes,ok,3605,22.257,True,True,1.000,1.642,0.871,0.871,2.484,16.000000,0.000,8.000,True,100.0%,0.250,True,0.100,0.742,balanced,80.0%,64,
8,dyads,9,✅ Yes,ok,4220,24.829,True,True,1.000,1.901,0.806,0.806,2.000,11.000000,0.000,8.000,True,100.0%,0.750,True,0.133,0.613,balanced,80.0%,64,
9,dyads,10,✅ Yes,ok,2468,13.942,True,True,1.000,1.642,0.871,0.871,2.484,16.000000,0.000,8.000,True,100.0%,0.250,True,0.100,0.742,balanced,80.0%,64,


### Observation
The dyad runs work fairly well overall. Most samples compile, the rhythm is usually correct, and the harmony stays in key most of the time. Because there are two notes at once, the lines move more and the changes are more frequent, so the texture feels busier than the single-note cases. Even with that added complexity, the results hold together reasonably well, with only a few rough edges.

##  Triads (3-note chords)

In [122]:
dfs = run_full_pipeline_for_all(only_batches=["triads"])

### Batch: **triads**  (n=20)

,batch,run,compiles,status,tokens,gen_s,bars_ok_musical,beats_ok,beats_per_bar_match_rate,interval_entropy,step_share,step_vs_leap,avg_interval_size,direction_changes,repeat_rate,note_density,in_key_ok,in_key_pct_time_weighted,tonal_stability_score,drift_detected,contour_diversity,ascending_tendency,style_label,adherence_text,note_count,bar_count_literal
0,triads,1,✅ Yes,ok,2740,18.906,True,True,1.000,1.828,0.871,0.871,1.968,16.000000,0.000,12.000,True,100.0%,0.500,True,0.133,0.613,balanced,80.0%,96,
1,triads,2,✅ Yes,ok,2130,13.805,True,True,1.000,1.828,0.871,0.871,1.968,16.000000,0.000,12.000,True,100.0%,0.500,True,0.133,0.613,balanced,80.0%,96,
2,triads,3,✅ Yes,ok,3526,23.623,True,True,1.000,1.482,1.000,1.000,0.968,4.000000,0.516,12.000,True,100.0%,1.000,False,0.200,0.533,balanced,80.0%,96,
3,triads,4,✅ Yes,ok,2406,15.273,True,True,1.000,1.642,0.871,0.871,2.484,8.000000,0.000,12.000,True,100.0%,0.500,True,0.133,0.742,balanced,80.0%,96,
4,triads,5,✅ Yes,ok,2497,17.980,True,True,1.000,1.157,0.871,0.871,3.000,8.000000,0.000,12.000,True,100.0%,0.500,True,0.100,0.871,balanced,80.0%,96,
5,triads,6,✅ Yes,ok,3733,25.696,True,True,1.000,1.157,0.871,0.871,3.000,8.000000,0.000,12.000,True,100.0%,0.500,True,0.100,0.871,balanced,80.0%,96,
6,triads,7,✅ Yes,ok,3511,21.495,True,True,1.000,2.388,0.806,0.806,2.065,15.000000,0.129,12.000,True,100.0%,1.000,False,0.267,0.444,balanced,80.0%,96,
7,triads,8,✅ Yes,ok,4145,29.391,True,True,1.000,2.630,0.774,0.774,1.871,10.000000,0.258,12.000,True,100.0%,0.250,True,0.267,0.522,balanced,80.0%,96,
8,triads,9,❌ No,empty,4522,37.328,,,,,,,,,,,,,,,,,,,,
9,triads,10,✅ Yes,ok,2480,16.427,True,True,1.000,2.063,0.774,0.774,2.258,19.000000,0.000,12.000,True,100.0%,0.500,True,0.133,0.516,balanced,80.0%,96,


### Observation
The triad samples are noticeably harder for the model. A good number still compile, but the error rate is higher, and the bar and beat structure is less consistent. With three notes stacked, the texture gets much busier and the lines change direction a lot, which makes things feel less controlled. They stay in key most of the time, but not as reliably as the dyads. Overall, the outputs are usable, just clearly less stable once full triads are involved.

-----------

## Conclusion
Across all the keys, modes, meters, and harmony settings, the model generally handles the basic cases well. Simple single-line melodies in common keys compile cleanly, stay in time, and keep their shape without much trouble. Things get a bit rougher in minor and modal settings, but still mostly hold together. As the tasks get heavier, faster movement, more complex meters, and especially chords, the outputs become busier and less consistent, with more missed details. Still, most of the results remain usable. Overall, the model is comfortable with straightforward melodic writing, and it starts to show limits only when the musical texture or harmony becomes more demanding.
